In [ ]:
# =============================================================================
#  Sampling uncertainty in key streamflow statistics as a function of
#  record length: a multi-climate bootstrap analysis
#  T.D. Muftuoglu
#
#  HOW TO RUN (Google Colab)
#    1. Download the daily discharge files of the stations from the GRDC Data
#       Portal (https://portal.grdc.bafg.de). Each station is delivered as a
#       text file named <GRDC-No>_Q_Day.Cmd.txt.
#    2. Run this cell. A "Choose Files" button appears below the cell;
#       select all station files at once and upload them.
#    3. When the run finishes, streamflow_uncertainty_outputs.zip (tables, figures and
#       replicate archive) is downloaded automatically.
#    Outside Colab, put the files in DATA_DIR and run the script with Python.
#
#  WORKFLOW
#    1. Read the GRDC files; keep complete water years (October-September);
#       treat -999 as missing and keep zero-flow days as observations.
#    2. Describe each catchment (flow statistics and hydrological signatures),
#       diagnose zero-flow days and test annual series for trends, change
#       points, variance shifts and serial correlation.
#    3. Resample whole water years to create synthetic short records of n
#       years with five schemes: random years without replacement, random
#       years with replacement, circular blocks of consecutive years, and all
#       contiguous n-year windows of the observed record.
#    4. Compare each short-record estimate with the full-record value using
#       relative, mean-flow-normalised, log-ratio and absolute errors; report
#       RMSE, bias and error percentiles.
#    5. Fit RMSE(n) = alpha * n^(-beta) with confidence intervals, for each
#       catchment and pooled across catchments.
#    6. Test how well the pooled relations transfer to an unseen catchment by
#       leave-one-catchment-out cross-validation.
#    7. Repeat the analysis without zero-flow days and with detrended series.
#    8. Apply the results to the short-record catchment (TARGET_ID).
#    9. Produce figures, tables and an archive of all replicate estimates.
# =============================================================================

# ── 0. Libraries ─────────────────────────────────────────────────────────────
# cartopy draws the station map; pymannkendall provides the Mann-Kendall tests;
# rasterio reads the Koppen-Geiger climate map. They are installed automatically
# if they are not available.
import subprocess, sys
for _pkg in ['cartopy', 'pymannkendall', 'rasterio', 'adjustText']:
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg])

import io, os, re, glob, json, zipfile, warnings, traceback, time, calendar
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
from scipy import stats

warnings.filterwarnings('ignore')

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import pymannkendall as pmk
    HAVE_PMK = True
except Exception:
    HAVE_PMK = False

try:
    from statsmodels.stats.diagnostic import acorr_ljungbox
    HAVE_SM = True
except Exception:
    HAVE_SM = False

# Figure style following the journal artwork guidelines: sans-serif lettering
# (Arial/Helvetica or the metrically identical Liberation Sans), lettering of
# at least 7 pt at the final printed width of 174 mm, and 600 dpi output.
def _sans_font():
    from matplotlib import font_manager
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in ['Arial', 'Helvetica', 'Liberation Sans']:
        if name in have:
            return name
    if IN_COLAB:
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-liberation'],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for f in glob.glob('/usr/share/fonts/truetype/liberation*/*.ttf'):
            font_manager.fontManager.addfont(f)
        if 'Liberation Sans' in {f.name for f in font_manager.fontManager.ttflist}:
            return 'Liberation Sans'
    return 'DejaVu Sans'

FIG_FONT = _sans_font()
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': [FIG_FONT, 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': FIG_FONT, 'mathtext.it': FIG_FONT + ':italic',
    'mathtext.bf': FIG_FONT + ':bold', 'mathtext.cal': FIG_FONT, 'mathtext.sf': FIG_FONT,
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8.5,
    'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5, 'legend.fontsize': 7.5,
    'figure.dpi': 110, 'savefig.dpi': 600,
    'savefig.bbox': 'tight', 'axes.spines.top': False, 'axes.spines.right': False,
})

OUT = 'outputs'
os.makedirs(OUT, exist_ok=True)
os.makedirs(f'{OUT}/supplementary', exist_ok=True)

# ── 1. Settings ──────────────────────────────────────────────────────────────
# All analysis choices are collected here so that they can be changed in one place.
SEED            = 2026                     # random seed (identical results on every run)
B               = 1000                     # replicates per scheme and record length
SAMPLE_LENGTHS  = [5, 8, 10, 12, 15, 20, 25, 30, 40, 50]   # synthetic record lengths n (years)
WATER_YEAR_START = 10                      # water year starts on 1 October (labelled by the end year)
COMPLETE_FRAC   = 0.95                     # a water year is used if >= 95% of its days are valid
BLOCK_LENGTHS   = [3, 5, 10]               # circular block bootstrap block lengths (years)
PRIMARY         = 'cbb5'                   # primary resampling scheme
MIN_WINDOWS     = 10                       # minimum number of contiguous windows per n
LOG_DELTA_FRAC  = 0.01                     # offset for log-ratio errors = 1% of mean flow
N_STATION_BOOT  = 1000                     # catchment bootstrap for pooled-fit CIs
N_TARGET_BOOT   = 2000                     # bootstrap of target-station scale parameter
TARGET_ID       = '6667500'                # short-record catchment (Cine Cayi)
DATA_DIR        = '.'                      # folder with the GRDC files (used only outside Colab)

# Station information used in tables and figures. Stations that are not listed
# here are still analysed; their names and coordinates are read from the file
# header and their climate is labelled 'Unassigned'.

STATION_META = {
    '6337590': {'river': 'Sieber',    'station': 'Hattorf',             'country': 'Germany',
                'lat': 51.651, 'lon': 10.255, 'area': 127.0,    'alt': 184.1, 'climate': 'Temperate oceanic'},
    '6122135': {'river': 'Remarde',   'station': 'St-Cyr-sous-Dourdan', 'country': 'France',
                'lat': 48.569, 'lon': 2.033,  'area': 147.0,    'alt': 77.2,  'climate': 'Temperate oceanic'},
    '6213800': {'river': 'Tagus',     'station': 'Trillo',              'country': 'Spain',
                'lat': 40.697, 'lon': -2.586, 'area': 3253.0,   'alt': 727.0, 'climate': 'Mediterranean/semi-arid'},
    '6233367': {'river': 'Segea',     'station': 'Svedala',             'country': 'Sweden',
                'lat': 55.507, 'lon': 13.238, 'area': 52.2,     'alt': None,  'climate': 'Boreal'},
    '6457850': {'river': 'Warta',     'station': 'Dzialoszyn',          'country': 'Poland',
                'lat': 51.110, 'lon': 18.873, 'area': 4093.6,   'alt': 172.6, 'climate': 'Continental'},
    '6444100': {'river': 'Tisa',      'station': 'Szeged',              'country': 'Hungary',
                'lat': 46.250, 'lon': 20.153, 'area': 138408.0, 'alt': 73.7,  'climate': 'Continental'},
    '6667500': {'river': 'Çine Çayı', 'station': 'Kayırlı',             'country': 'Turkey',
                'lat': 37.420, 'lon': 28.120, 'area': 948.0,    'alt': 262.0, 'climate': 'Mediterranean'},
}
# Additional reference stations (names used in figures and tables).
_EXTRA = [
    ('6227620', 'Mijares', 'El Terde', 'Spain'), ('6227160', 'Alfambra', 'Villalba Alta', 'Spain'),
    ('6226460', 'Jalón', 'Jubera', 'Spain'), ('6212780', 'Ucero', 'Osma', 'Spain'),
    ('6212720', 'Arlanza', 'Salas de los Infantes', 'Spain'), ('6213540', 'Tiétar', 'Arenas de San Pedro', 'Spain'),
    ('6213510', 'Alagón', 'Garcibuey', 'Spain'), ('6226470', 'Arba de Luesia', 'Biota', 'Spain'),
    ('6213790', 'Badiel', 'Torre del Burgo', 'Spain'), ('6128035', 'Tech', 'Amélie-les-Bains', 'France'),
    ('6128040', 'Têt', 'Serdinya', 'France'), ('6128095', 'Hérault', 'Laroque', 'France'),
    ('6128630', 'Réal Martin', 'La Crau', 'France'), ('6128145', 'Argens', 'Châteauvert', 'France'),
    ('6196010', 'Xeros', 'Lazarides', 'Cyprus'), ('6196045', 'Peristerona', 'Panagia Bridge', 'Cyprus'),
    ('6196350', 'Germasogeia', 'Phinikaria', 'Cyprus')]
for _sid, _r, _st, _c in _EXTRA:
    STATION_META.setdefault(_sid, {'river': _r, 'station': _st, 'country': _c,
                                   'lat': None, 'lon': None, 'area': None, 'alt': None,
                                   'climate': 'Mediterranean'})

# Koppen-Geiger class at each gauge from the 1-km present-day map of Beck et al.
# (2018), downloaded from figshare. If the download fails, the 'climate' entries
# above are used instead.
USE_KOPPEN_MAP  = True
KOPPEN_ARTICLE  = 6396959
KG_LEGEND = ['Af', 'Am', 'Aw', 'BWh', 'BWk', 'BSh', 'BSk', 'Csa', 'Csb', 'Csc', 'Cwa', 'Cwb', 'Cwc',
             'Cfa', 'Cfb', 'Cfc', 'Dsa', 'Dsb', 'Dsc', 'Dsd', 'Dwa', 'Dwb', 'Dwc', 'Dwd', 'Dfa', 'Dfb',
             'Dfc', 'Dfd', 'ET', 'EF']
GROUP_ORDER = ['Mediterranean (Cs)', 'Semi-arid (B)', 'Temperate (Cf)', 'Continental (D)', 'Other']
GROUP_COL = {'Mediterranean (Cs)': '#d7301f', 'Semi-arid (B)': '#fe9929', 'Temperate (Cf)': '#2b8cbe',
             'Continental (D)': '#6a51a3', 'Other': '#969696'}
TARGET_COL = '#d01c8b'
FALLBACK_GROUP = {'Temperate oceanic': 'Temperate (Cf)', 'Mediterranean/semi-arid': 'Mediterranean (Cs)',
                  'Mediterranean': 'Mediterranean (Cs)', 'Boreal': 'Continental (D)',
                  'Continental': 'Continental (D)'}
COUNTRY_NAMES = {'DE': 'Germany', 'FR': 'France', 'ES': 'Spain', 'SE': 'Sweden', 'PL': 'Poland',
                 'HU': 'Hungary', 'TR': 'Turkey', 'CY': 'Cyprus', 'IT': 'Italy', 'PT': 'Portugal',
                 'GR': 'Greece', 'IL': 'Israel'}


STAT_KEYS  = ['q_mean', 'q10', 'q50', 'q90', 'cv', 'ann_max', 'q100']
STAT_TEX   = {'q_mean': r'$\bar{Q}$', 'q10': r'$Q_{10}$', 'q50': r'$Q_{50}$',
              'q90': r'$Q_{90}$', 'cv': 'CV', 'ann_max': r'$\overline{Q}_{\max}$',
              'q100': r'$Q_{100}$ (Gumbel)'}
STAT_TXT   = {'q_mean': 'Qmean', 'q10': 'Q10', 'q50': 'Q50', 'q90': 'Q90',
              'cv': 'CV', 'ann_max': 'AnnMax', 'q100': 'Q100_Gumbel'}
FLOW_STATS = [k for k in STAT_KEYS if k != 'cv']
MEASURES   = ['rel', 'norm', 'log', 'abs']
SI = {k: i for i, k in enumerate(STAT_KEYS)}
NI = {n: j for j, n in enumerate(SAMPLE_LENGTHS)}
SCHEME_LABEL = {'sub': 'Subsampling (without replacement)',
                'sub_fpc': 'Subsampling, FPC-corrected',
                'iid': 'Bootstrap, independent years',
                'cbb3': 'Block bootstrap, L = 3 yr',
                'cbb5': 'Block bootstrap, L = 5 yr',
                'cbb10': 'Block bootstrap, L = 10 yr',
                'win': 'Contiguous windows'}
SCHEME_STYLE = {'sub': dict(color='#7f7f7f', ls='-', marker='s'),
                'sub_fpc': dict(color='#7f7f7f', ls=':', marker='x'),
                'iid': dict(color='#1f77b4', ls='-', marker='o'),
                'cbb3': dict(color='#9ecae1', ls='-', marker='^'),
                'cbb5': dict(color='#d62728', ls='-', marker='D'),
                'cbb10': dict(color='#ff9896', ls='-', marker='v'),
                'win': dict(color='#2ca02c', ls='--', marker='^')}


def log(msg):
    print(msg, flush=True)


def water_year(index):
    """Water-year label of each date (the year in which the water year ends)."""
    y = index.year.values
    return y + (index.month.values >= WATER_YEAR_START).astype(int) if WATER_YEAR_START > 1 else y


def days_in_water_year(wy):
    """Number of days in a water year; February falls in the labelled year."""
    return 366 if calendar.isleap(wy) else 365


# ── 2. Reading the GRDC files ────────────────────────────────────────────────
# A GRDC file has a header (lines starting with '#') with the station metadata,
# followed by one line per day: date; time; discharge (m3/s). Missing days are
# coded as -999.000.
def parse_grdc(raw_bytes):
    """Read one GRDC file; return header metadata, a continuous daily series
    (NaN for missing days) and the number of days coded as missing."""
    text = raw_bytes.decode('latin-1', errors='replace')
    lines = text.splitlines()
    hdr = [l for l in lines if l.startswith('#')]

    def h(pattern):
        rx = re.compile(r'^#\s*' + pattern + r'\s*:\s*(.*)$', re.I)
        for l in hdr:
            m = rx.match(l)
            if m:
                return m.group(1).strip()
        return None

    def f(x):
        try:
            v = float(x)
            return None if v <= -999 else v
        except Exception:
            return None

    meta = {'grdc_no': h(r'GRDC-No\.'), 'river': h('River'), 'station': h('Station'),
            'country': h('Country'), 'lat': f(h(r'Latitude \(DD\)')),
            'lon': f(h(r'Longitude \(DD\)')), 'area': f(h(r'Catchment area \(km[^)]*\)')),
            'alt': f(h(r'Altitude \(m ASL\)'))}
    body = [l for l in lines if l.strip() and not l.startswith('#')]
    df = pd.read_csv(io.StringIO('\n'.join(body)), sep=';', skipinitialspace=True, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    dcol = [c for c in df.columns if 'YYYY' in c.upper() or c.lower().startswith('date')][0]
    vcol = None
    for cand in ['value', 'original', 'calculated']:
        for c in df.columns:
            if c.lower() == cand:
                vcol = c
                break
        if vcol:
            break
    if vcol is None:
        for c in df.columns:
            if c == dcol or 'hh' in c.lower():
                continue
            if pd.to_numeric(df[c], errors='coerce').notna().sum() > 100:
                vcol = c
                break
    q = pd.to_numeric(df[vcol].str.strip(), errors='coerce').values
    d = pd.to_datetime(df[dcol].str.strip(), errors='coerce')
    s = pd.Series(q, index=d)
    s = s[s.index.notna()]
    s = s[~s.index.duplicated()].sort_index()
    n_code = int((s < 0).sum())
    s[s < 0] = np.nan
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq='D'))
    return meta, s, n_code


def load_files():
    """Ask for the station files in Colab, or read them from DATA_DIR."""
    out = {}
    if IN_COLAB:
        log('Click "Choose Files" below and select all GRDC station files (*_Q_Day.Cmd.txt).')
        up = colab_files.upload()
        items = list(up.items())
    else:
        items = [(os.path.basename(p), open(p, 'rb').read())
                 for p in sorted(glob.glob(os.path.join(DATA_DIR, '*.txt')))]
    for fname, fbytes in items:
        head = fbytes[:3000].decode('latin-1', errors='replace').upper()
        if 'Q_DAY' not in fname.upper() and 'DAILY' not in head:
            log(f'  skipped (not a daily discharge file): {fname}')
            continue
        meta, s, n_code = parse_grdc(fbytes)
        sid = (meta.get('grdc_no') or fname.split('_')[0]).strip()
        out[sid] = (meta, s, n_code)
    return out


def koppen_lookup(points):
    """Koppen-Geiger symbol at each (lon, lat) point. If the gauge pixel has no
    class (e.g. on a coast line), the most frequent class in the surrounding
    5 x 5 pixels is used."""
    import requests
    import rasterio
    from rasterio.windows import Window
    files = requests.get(f'https://api.figshare.com/v2/articles/{KOPPEN_ARTICLE}/files', timeout=60).json()
    f = [x for x in files if x['name'].lower().endswith('.zip')][0]
    if not os.path.exists('koppen.zip'):
        with requests.get(f['download_url'], stream=True, timeout=600) as r:
            r.raise_for_status()
            with open('koppen.zip', 'wb') as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
    with zipfile.ZipFile('koppen.zip') as zf:
        names = zf.namelist()
        tifs = [n for n in names if 'present' in n.lower() and n.lower().endswith('.tif')]
        tif = ([n for n in tifs if '0p0083' in n] or sorted(tifs))[0]
        zf.extract(tif, 'koppen')
        legend = list(KG_LEGEND)
        leg = [n for n in names if n.lower().endswith('legend.txt')]
        if leg:
            txt = zf.read(leg[0]).decode('latin-1')
            found = re.findall(r'^\s*(\d+):\s+([A-Z][A-Za-z]{1,2})\b', txt, re.M)
            if len(found) >= 30:
                legend = [c for _, c in sorted(found, key=lambda t: int(t[0]))]
    out = []
    with rasterio.open(os.path.join('koppen', tif)) as src:
        for lon, lat in points:
            r_, c_ = src.index(lon, lat)
            w = src.read(1, window=Window(c_ - 2, r_ - 2, 5, 5), boundless=True, fill_value=0)
            v = int(w[2, 2])
            if v == 0:
                vals = w[w > 0]
                v = int(np.bincount(vals).argmax()) if len(vals) else 0
            out.append(legend[v - 1] if 1 <= v <= len(legend) else None)
    return out


def koppen_group(code):
    if not code:
        return None
    if code.startswith('Cs'):
        return 'Mediterranean (Cs)'
    if code.startswith('B'):
        return 'Semi-arid (B)'
    if code.startswith('C'):
        return 'Temperate (Cf)'
    if code.startswith('D'):
        return 'Continental (D)'
    return 'Other'


# ── 3. Streamflow statistics and resampling ──────────────────────────────────
# Seven statistics are computed for every record (full or synthetic):
#   q_mean  mean daily discharge
#   q10     discharge exceeded 10% of the time (high flow)
#   q50     median discharge
#   q90     discharge exceeded 90% of the time (low flow)
#   cv      coefficient of variation of daily discharge
#   ann_max mean of the annual maximum daily discharges
#   q100    100-year flood from a Gumbel distribution fitted by L-moments
# Flow-duration-curve quantiles use the Weibull plotting position i/(n+1).
def gumbel_quantile(x, T=100.0):
    """Gumbel quantile from sample L-moments (Hosking and Wallis 1997)."""
    x = np.sort(np.asarray(x, float))
    n = len(x)
    b0 = x.mean()
    if n < 2:
        return b0
    j = np.arange(1, n + 1)
    b1 = np.sum((j - 1) / (n - 1) * x) / n
    l2 = 2 * b1 - b0
    a = l2 / np.log(2.0)
    xi = b0 - 0.5772156649 * a
    return xi - a * np.log(-np.log(1.0 - 1.0 / T))


class YearSet:
    """Daily flows of the complete water years of one station, stored year by
    year, with pre-computed yearly sums so that statistics of any combination
    of years (including repeated years) are obtained quickly."""

    def __init__(self, arrays, years):
        self.Y = [np.asarray(a, float) for a in arrays]
        self.years = np.asarray(years)
        self.N = len(self.Y)
        self.S = np.array([a.sum() for a in self.Y])
        self.C = np.array([len(a) for a in self.Y], float)
        self.SS = np.array([(a ** 2).sum() for a in self.Y])
        self.AM = np.array([a.max() if len(a) else np.nan for a in self.Y])

    def stats(self, idx):
        idx = np.asarray(idx, int)
        c = self.C[idx].sum()
        m = self.S[idx].sum() / c
        var = max(self.SS[idx].sum() / c - m * m, 0.0)
        cv = np.sqrt(var) / m if m > 0 else np.nan
        x = np.concatenate([self.Y[i] for i in idx])
        q_lo, q_md, q_hi = np.quantile(x, [0.10, 0.50, 0.90], method='weibull')
        am = self.AM[idx]
        return (m, q_hi, q_md, q_lo, cv, am.mean(), gumbel_quantile(am, 100.0))

    def reference(self):
        return np.array(self.stats(np.arange(self.N)))

    def annual_series(self):
        """Single-year values of each statistic (for scale parameters)."""
        a = np.array([self.stats([i]) for i in range(self.N)])
        a[:, SI['q100']] = self.AM
        return a


def draw_indices(scheme, N, n, rng):
    """Positions of the water years that form one synthetic n-year record.
    sub  : n different years drawn at random (without replacement)
    iid  : n years drawn at random with replacement
    cbbL : blocks of L consecutive years starting at random positions; the
           record is treated as circular so that every year is equally likely"""
    if scheme == 'sub':
        return rng.choice(N, n, replace=False)
    if scheme == 'iid':
        return rng.integers(0, N, n)
    if scheme.startswith('cbb'):
        L = int(scheme[3:])
        k = int(np.ceil(n / L))
        st = rng.integers(0, N, k)
        return ((st[:, None] + np.arange(L)[None, :]) % N).ravel()[:n]
    raise ValueError(scheme)


def run_scheme(ys, scheme, rng):
    """Estimates of all statistics for B synthetic records at every n.
    'win' uses every contiguous n-year window of the record instead of random
    draws (only when at least MIN_WINDOWS windows exist)."""
    N = ys.N
    est = np.full((len(STAT_KEYS), len(SAMPLE_LENGTHS), B), np.nan)
    for j, n in enumerate(SAMPLE_LENGTHS):
        if scheme == 'win':
            nw = N - n + 1
            if nw < MIN_WINDOWS:
                continue
            for b, s0 in enumerate(range(min(nw, B))):
                est[:, j, b] = ys.stats(np.arange(s0, s0 + n))
            continue
        if scheme == 'sub' and n >= N:
            continue
        for b in range(B):
            est[:, j, b] = ys.stats(draw_indices(scheme, N, n, rng))
    return est


def log_delta(stat, qbar):
    """Small offset (1% of mean flow) that keeps log-ratios finite at zero flow."""
    return 0.0 if stat == 'cv' else LOG_DELTA_FRAC * qbar


def errors(est, ref, qbar, stat, measure):
    """est: array of estimates; ref: reference value of the statistic."""
    if measure == 'rel':
        return (est - ref) / ref * 100.0 if ref > 0 else np.full_like(est, np.nan)
    if measure == 'norm':
        if stat == 'cv':
            return (est - ref) / ref * 100.0 if ref > 0 else np.full_like(est, np.nan)
        return (est - ref) / qbar * 100.0
    if measure == 'log':
        # a statistic whose full-record value is zero (e.g. the low-flow index of
        # a river that is dry more than 10% of the time) has no meaningful ratio
        d = log_delta(stat, qbar)
        if ref <= 0 or ref + d <= 0:
            return np.full_like(est, np.nan)
        with np.errstate(divide='ignore', invalid='ignore'):
            e = np.log((est + d) / (ref + d))
        e[~np.isfinite(e)] = np.nan
        return e
    if measure == 'abs':
        return est - ref
    raise ValueError(measure)


def to_pct(measure, v):
    """Log-ratio values are converted to percentage equivalents 100(exp(v)-1)."""
    return 100.0 * (np.exp(v) - 1.0) if measure == 'log' else v


def rmse_of(e):
    """Root-mean-square of the finite errors."""
    e = e[np.isfinite(e)]
    return np.sqrt(np.mean(e ** 2)) if len(e) >= 5 else np.nan


def summarise(e, measure):
    """RMSE, mean bias (with 95% interval), median and percentiles of errors."""
    e = e[np.isfinite(e)]
    if len(e) < 5:
        return None
    r = np.sqrt(np.mean(e ** 2))
    mb = e.mean()
    se = e.std(ddof=1) / np.sqrt(len(e))
    q = np.percentile(e, [2.5, 5, 25, 50, 75, 95, 97.5])
    return {'n_rep': len(e), 'RMSE': to_pct(measure, r), 'RMSE_raw': r,
            'mean_bias': to_pct(measure, mb), 'mean_bias_lo': to_pct(measure, mb - 1.96 * se),
            'mean_bias_hi': to_pct(measure, mb + 1.96 * se), 'median': to_pct(measure, q[3]),
            'SD': e.std(ddof=1), 'P2.5': to_pct(measure, q[0]), 'P5': to_pct(measure, q[1]),
            'P25': to_pct(measure, q[2]), 'P75': to_pct(measure, q[4]),
            'P95': to_pct(measure, q[5]), 'P97.5': to_pct(measure, q[6]),
            'width_P5_P95': to_pct(measure, q[5]) - to_pct(measure, q[1])}


# ── 4. Power-law fitting ─────────────────────────────────────────────────────
# RMSE(n) = alpha * n^(-beta) is fitted by non-linear least squares. Standard
# errors come from the parameter covariance matrix; 95% intervals use the
# t distribution with k - 2 degrees of freedom (k = number of record lengths).
def power_law(n, a, b):
    """RMSE(n) = a * n^(-b)."""
    return a * np.power(n, -b)


def fit_pl(ns, rs):
    """Fit the power law and return parameters, standard errors, 95% intervals and R2."""
    ns = np.asarray(ns, float)
    rs = np.asarray(rs, float)
    ok = np.isfinite(rs) & (rs > 0)
    ns, rs = ns[ok], rs[ok]
    if len(ns) < 4:
        return None
    try:
        p0 = [rs[0] * ns[0] ** 0.5, 0.5]
        popt, pcov = curve_fit(power_law, ns, rs, p0=p0, maxfev=20000)
    except Exception:
        return None
    pred = power_law(ns, *popt)
    ss_res = np.sum((rs - pred) ** 2)
    ss_tot = np.sum((rs - rs.mean()) ** 2)
    se = np.sqrt(np.diag(pcov)) if np.all(np.isfinite(pcov)) else np.array([np.nan, np.nan])
    tq = stats.t.ppf(0.975, len(ns) - 2)
    return {'alpha': popt[0], 'beta': popt[1], 'R2': 1 - ss_res / ss_tot if ss_tot > 0 else np.nan,
            'se_alpha': se[0], 'se_beta': se[1],
            'alpha_lo': popt[0] - tq * se[0], 'alpha_hi': popt[0] + tq * se[0],
            'beta_lo': popt[1] - tq * se[1], 'beta_hi': popt[1] + tq * se[1], 'k': len(ns)}


def pl_at(fit, n):
    """Evaluate a fitted power law at record length n."""
    return power_law(n, fit['alpha'], fit['beta']) if fit else np.nan


# ── 5. Catchment signatures and statistical tests ────────────────────────────
def lyne_hollick_bfi(q, a=0.925, passes=3):
    """Baseflow index from the Lyne-Hollick digital filter (three passes)."""
    q = np.asarray(q, float)
    if len(q) < 10 or np.nansum(q) <= 0:
        return np.nan
    b = q.copy()
    for p in range(passes):
        x = b if p % 2 == 0 else b[::-1]
        qf = np.zeros_like(x)
        for t in range(1, len(x)):
            qf[t] = a * qf[t - 1] + 0.5 * (1 + a) * (x[t] - x[t - 1])
            qf[t] = min(max(qf[t], 0.0), x[t])
        bf = x - qf
        b = bf if p % 2 == 0 else bf[::-1]
    return b.sum() / q.sum()


def mk_original(x):
    """Mann-Kendall trend test with tie correction (used if pymannkendall is missing)."""
    x = np.asarray(x, float)
    n = len(x)
    s = 0.0
    for k in range(n - 1):
        s += np.sign(x[k + 1:] - x[k]).sum()
    _, tc = np.unique(x, return_counts=True)
    var = (n * (n - 1) * (2 * n + 5) - np.sum(tc * (tc - 1) * (2 * tc + 5))) / 18.0
    z = (s - np.sign(s)) / np.sqrt(var) if s != 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    tau = s / (0.5 * n * (n - 1))
    return tau, p


def sen_slope(x):
    """Sen slope (median of pairwise slopes) and intercept."""
    x = np.asarray(x, float)
    n = len(x)
    sl = [(x[j] - x[i]) / (j - i) for i in range(n - 1) for j in range(i + 1, n)]
    b = np.median(sl)
    a = np.median(x - b * np.arange(n))
    return b, a


def pettitt(x):
    """Pettitt change-point test: position of the most likely change and p-value."""
    x = np.asarray(x, float)
    n = len(x)
    U = np.array([np.sign(x[:t + 1, None] - x[None, t + 1:]).sum() for t in range(n - 1)])
    t = int(np.argmax(np.abs(U)))
    K = np.abs(U[t])
    p = min(1.0, 2 * np.exp(-6 * K ** 2 / (n ** 3 + n ** 2)))
    return t + 1, p          # index of first year after the change


def lag1(x):
    """Lag-1 autocorrelation coefficient."""
    x = np.asarray(x, float) - np.mean(x)
    return np.sum(x[1:] * x[:-1]) / np.sum(x ** 2)


def trend_tests(x, years):
    """All stationarity and persistence tests for one annual series."""
    x = np.asarray(x, float)
    n = len(x)
    res = {}
    if HAVE_PMK:
        r = pmk.original_test(x)
        res['MK_tau'], res['MK_p'] = r.Tau, r.p
        try:
            r2 = pmk.hamed_rao_modification_test(x)
            res['MKmod_p'] = r2.p
        except Exception:
            res['MKmod_p'] = np.nan
    else:
        res['MK_tau'], res['MK_p'] = mk_original(x)
        res['MKmod_p'] = np.nan
    b, a = sen_slope(x)
    res['Sen_slope_pct_per_decade'] = 10 * b / np.mean(x) * 100 if np.mean(x) > 0 else np.nan
    k, p = pettitt(x)
    res['Pettitt_year'] = int(years[k]) if k < n else np.nan
    res['Pettitt_p'] = p
    res['Levene_p'] = stats.levene(x[:n // 2], x[n // 2:]).pvalue
    r1 = lag1(x)
    res['r1'] = r1
    res['r1_bound95'] = (-1 + 1.96 * np.sqrt(n - 2)) / (n - 1)
    if HAVE_SM:
        try:
            lb = acorr_ljungbox(x - x.mean(), lags=[5], return_df=True)
            res['LjungBox5_p'] = float(lb['lb_pvalue'].iloc[0])
        except Exception:
            res['LjungBox5_p'] = np.nan
    else:
        res['LjungBox5_p'] = np.nan
    return res


def zero_diagnostics(daily):
    """Describe zero-flow days: how many, how long the zero runs last, in which
    season they occur and whether they follow a recession (physically plausible
    no-flow periods) or appear as isolated single days (possible recording errors)."""
    q = daily.dropna()
    z = (q == 0).values
    out = {'valid_days': len(q), 'zero_days': int(z.sum()),
           'zero_frac_pct': 100 * z.mean() if len(q) else np.nan}
    if z.sum() == 0:
        out.update({'zero_runs': 0, 'run_median_d': np.nan, 'run_max_d': np.nan,
                    'isolated_zero_pct': np.nan, 'zero_in_JJAS_pct': np.nan,
                    'years_with_zero': 0, 'pre_run_Q_over_Q50_median': np.nan})
        return out
    runs, start = [], None
    for i, v in enumerate(z):
        if v and start is None:
            start = i
        if (not v) and start is not None:
            runs.append((start, i - start))
            start = None
    if start is not None:
        runs.append((start, len(z) - start))
    lens = np.array([r[1] for r in runs])
    months = q.index.month[z]
    q50 = np.median(q.values)
    pre = [q.values[s - 1] / q50 for s, _ in runs if s > 0 and q50 > 0]
    out.update({'zero_runs': len(runs), 'run_median_d': float(np.median(lens)),
                'run_max_d': int(lens.max()),
                'isolated_zero_pct': 100 * np.sum(lens == 1) / z.sum(),
                'zero_in_JJAS_pct': 100 * np.isin(months, [6, 7, 8, 9]).mean(),
                'years_with_zero': int(pd.Series(z, index=water_year(q.index)).groupby(level=0).any().sum()),
                'pre_run_Q_over_Q50_median': float(np.median(pre)) if pre else np.nan})
    return out


def signatures(daily, ys):
    """Hydrological signatures used to describe and compare catchments:
    variability of daily and annual flows, baseflow index, fraction of zero-flow
    days, low-flow ratio, slope of the flow-duration curve, strength and
    timing of seasonality, flashiness and lag-1 autocorrelation of annual flows."""
    q = daily.dropna()
    qv = q.values
    qbar = qv.mean()
    ann = ys.S / ys.C
    q33, q66 = np.quantile(qv, [1 - 0.33, 1 - 0.66], method='weibull')
    d = LOG_DELTA_FRAC * qbar
    doy = q.index.dayofyear.values
    th = 2 * np.pi * (doy - 0.5) / 365.25
    C_, S_ = np.sum(qv * np.cos(th)), np.sum(qv * np.sin(th))
    R = np.hypot(C_, S_) / qv.sum()
    timing = (np.arctan2(S_, C_) % (2 * np.pi)) * 365.25 / (2 * np.pi)
    bfi = np.nanmean([lyne_hollick_bfi(a) for a in ys.Y])
    rb = np.nanmean([np.abs(np.diff(a)).sum() / a.sum() if a.sum() > 0 else np.nan for a in ys.Y])
    return {'Qbar': qbar, 'CV_daily': qv.std() / qbar, 'CV_annual': ann.std(ddof=1) / ann.mean(),
            'BFI': bfi, 'zero_frac': (qv == 0).mean(),
            'Q90_over_Qbar': np.quantile(qv, 0.10, method='weibull') / qbar,
            'FDC_slope': (np.log(q33 + d) - np.log(q66 + d)) / 0.33,
            'Seasonality_R': R, 'Timing_day': timing, 'Flashiness_RB': rb,
            'r1_annual': lag1(ann)}


SIG_FOR_DISTANCE = ['CV_annual', 'CV_daily', 'BFI', 'zero_frac', 'Q90_over_Qbar',
                    'FDC_slope', 'Seasonality_R', 'Flashiness_RB']


def sig_matrix(sig_df, ids):
    """Standardised (z-score) signature matrix for the given stations."""
    X = sig_df.loc[ids, SIG_FOR_DISTANCE].astype(float).copy()
    tim = sig_df.loc[ids, 'Timing_day'].astype(float) * 2 * np.pi / 365.25
    X['t_cos'] = np.cos(tim) * sig_df.loc[ids, 'Seasonality_R']
    X['t_sin'] = np.sin(tim) * sig_df.loc[ids, 'Seasonality_R']
    sd = X.std(ddof=0).replace(0, 1.0)
    return (X - X.mean()) / sd


def nearest_donor(sig_df, target, candidates):
    """Euclidean distance between the target and each candidate in signature space."""
    Z = sig_matrix(sig_df, list(candidates) + [target])
    d = np.sqrt(((Z.loc[candidates] - Z.loc[target]) ** 2).sum(axis=1))
    return d.sort_values()


def scale_param(annual_vals, stat, measure, qbar):
    """Variability S of the single-year values of a statistic, used by the
    scaled model. For the mean, S is the coefficient of variation of annual
    mean flow, and RMSE = 100 * S / sqrt(n) for independent years."""
    x = np.asarray(annual_vals, float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return np.nan
    if measure == 'rel':
        return x.std(ddof=1) / x.mean() if x.mean() > 0 else np.nan
    if measure == 'norm':
        return x.std(ddof=1) / (x.mean() if stat == 'cv' else qbar)
    if measure == 'log':
        d = log_delta(stat, qbar)
        if np.any(x + d <= 0):
            return np.nan
        return np.log(x + d).std(ddof=1)
    return x.std(ddof=1)


# =============================================================================
#  ANALYSIS
# =============================================================================
T0 = time.time()
raw = load_files()
if not raw:
    raise SystemExit('No GRDC files found.')

# Station metadata: names from STATION_META, coordinates and areas from the file
# header when they are not given above.
META = {}
for sid, (hm, s, ncode) in sorted(raw.items()):
    m = dict(STATION_META.get(sid, {}))
    for k in ['river', 'station', 'country', 'lat', 'lon', 'area', 'alt']:
        if m.get(k) in (None, '') and hm.get(k) not in (None, ''):
            m[k] = hm[k].title() if isinstance(hm[k], str) and k in ('river', 'station') else hm[k]
    m['country'] = COUNTRY_NAMES.get(m.get('country'), m.get('country'))
    m.setdefault('climate', 'Unassigned')
    META[sid] = m

ORDER = [s for s in STATION_META if s in raw] + sorted(s for s in raw if s not in STATION_META)
REF_IDS = [s for s in ORDER if s != TARGET_ID]
HAS_TARGET = TARGET_ID in raw
log(f'\nLoaded {len(raw)} stations; reference set = {len(REF_IDS)}; target present = {HAS_TARGET}')

# Climate class of each station
KOPPEN_OK = False
if USE_KOPPEN_MAP:
    try:
        log('Reading the Koppen-Geiger map (Beck et al. 2018) ...')
        codes = koppen_lookup([(META[s]['lon'], META[s]['lat']) for s in ORDER])
        for sid, c in zip(ORDER, codes):
            META[sid]['koppen'] = c
        KOPPEN_OK = True
    except Exception as e:
        log(f'  Koppen map not available ({e}); climate labels from STATION_META are used.')
GROUP = {}
for sid in ORDER:
    g = koppen_group(META[sid].get('koppen')) if KOPPEN_OK else None
    GROUP[sid] = g or FALLBACK_GROUP.get(META[sid]['climate'], 'Other')
    META[sid]['group'] = GROUP[sid]
COLORS = {sid: (TARGET_COL if sid == TARGET_ID else GROUP_COL[GROUP[sid]]) for sid in ORDER}
GROUPS_PRESENT = [g for g in GROUP_ORDER if any(GROUP[s] == g for s in REF_IDS)]
# Station numbers used on the map and in Table 1: reference stations ordered by
# climate group, country and river name; the short-record station is last.
_ref_sorted = sorted(REF_IDS, key=lambda s: (GROUP_ORDER.index(GROUP[s]), META[s]['country'], META[s]['river']))
NUMBER = {sid: i + 1 for i, sid in enumerate(_ref_sorted + ([TARGET_ID] if HAS_TARGET else []))}
for sid in ORDER:
    log(f"  {sid}  {META[sid]['river']:16s} {META[sid]['country']:9s} "
        f"Koppen = {META[sid].get('koppen')}  group = {GROUP[sid]}")

# ── Data screening ───────────────────────────────────────────────────────────
# Only complete water years are used: a year is complete when at least
# COMPLETE_FRAC of its days have a valid value. Partial first and last years of
# a record are therefore removed. Zero discharge is a valid observation.
DAILY, YS, YS_NZ, SCREEN = {}, {}, {}, []
for sid in ORDER:
    hm, s, ncode = raw[sid]
    wy = water_year(s.index)
    years_all = np.unique(wy)
    nvalid = s.notna().groupby(wy).sum()
    ndays = pd.Series({y: days_in_water_year(y) for y in years_all})
    frac = (nvalid / ndays).reindex(years_all).fillna(0)
    comp = frac.index[frac >= COMPLETE_FRAC].values
    d = s[np.isin(wy, comp)]
    DAILY[sid] = d
    dwy = water_year(d.index)
    arrs = [d[dwy == y].dropna().values for y in comp]
    YS[sid] = YearSet(arrs, comp)
    YS_NZ[sid] = YearSet([a[a > 0] for a in arrs], comp)
    SCREEN.append({'GRDC_No': sid, 'River': META[sid]['river'], 'First_year': int(years_all.min()),
                   'Last_year': int(years_all.max()), 'Water_years_in_file': len(years_all),
                   'Complete_years_used': len(comp), 'First_complete': int(comp.min()) if len(comp) else None,
                   'Last_complete': int(comp.max()) if len(comp) else None,
                   'Gap_years_inside_period': int((comp.max() - comp.min() + 1) - len(comp)) if len(comp) else None,
                   'Missing_code_days_in_file': ncode,
                   'Missing_days_in_used_years': int(d.isna().sum()),
                   'Zero_days_in_used_years': int((d == 0).sum()),
                   'Mean_valid_frac_used_years_pct': 100 * float(frac.loc[comp].mean()) if len(comp) else None})
    log(f"  {META[sid]['river']:12s}: {len(years_all)} water years in file, "
        f"{len(comp)} complete ({comp.min()}-{comp.max()}), zero days = {int((d == 0).sum())}, "
        f"missing-coded days = {ncode}")
T_SCREEN = pd.DataFrame(SCREEN)
T_SCREEN.to_csv(f'{OUT}/supplementary/TableS1_screening.csv', index=False)

REF = {sid: YS[sid].reference() for sid in ORDER}
REF_NZ = {sid: YS_NZ[sid].reference() for sid in ORDER}
QBAR = {sid: REF[sid][SI['q_mean']] for sid in ORDER}
ANNUAL = {sid: YS[sid].annual_series() for sid in ORDER}
NYR = {sid: YS[sid].N for sid in ORDER}
N_TARGET = NYR[TARGET_ID] if HAS_TARGET else 11

# ── Zero-flow diagnostics ────────────────────────────────────────────────────
T_ZERO = pd.DataFrame([{'GRDC_No': sid, 'River': META[sid]['river'], **zero_diagnostics(DAILY[sid])}
                       for sid in ORDER])
T_ZERO.to_csv(f'{OUT}/supplementary/TableS2_zero_flow_diagnostics.csv', index=False)
log('\nZero-flow diagnostics'); log(T_ZERO.to_string(index=False))

# ── Station table with flow statistics and hydrological signatures ──────────
SIG = pd.DataFrame({sid: signatures(DAILY[sid], YS[sid]) for sid in ORDER}).T
rows = []
for sid in ORDER:
    m, r = META[sid], REF[sid]
    rows.append({'No': NUMBER[sid], 'GRDC_No': sid, 'River': m['river'], 'Station': m['station'], 'Country': m['country'],
                 'Koppen': m.get('koppen'), 'Climate_group': GROUP[sid], 'Lat': m['lat'], 'Lon': m['lon'], 'Area_km2': m['area'],
                 'Alt_m': m['alt'], 'First_complete': int(YS[sid].years.min()),
                 'Last_complete': int(YS[sid].years.max()), 'N_years': NYR[sid],
                 'Qbar': r[SI['q_mean']], 'Q10': r[SI['q10']], 'Q50': r[SI['q50']], 'Q90': r[SI['q90']],
                 'Qmin': float(np.nanmin(DAILY[sid].values)), 'Qmax': float(np.nanmax(DAILY[sid].values)),
                 'CV_daily': r[SI['cv']], 'AnnMax_mean': r[SI['ann_max']], 'Q100_Gumbel': r[SI['q100']],
                 **{k: SIG.loc[sid, k] for k in ['CV_annual', 'BFI', 'zero_frac', 'Seasonality_R',
                                                  'Timing_day', 'Flashiness_RB', 'FDC_slope', 'r1_annual']}})
T1 = pd.DataFrame(rows).sort_values('No')
T1.to_csv(f'{OUT}/Table1_stations_signatures.csv', index=False)
log('\nTable 1'); log(T1.round(3).to_string(index=False))

# ── Stationarity and persistence of the annual series ───────────────────────
# Annual mean, annual maximum and annual minimum 7-day mean flow are tested for
# monotonic trend (Mann-Kendall, also with the Hamed-Rao correction for
# autocorrelation; Sen slope), abrupt change (Pettitt), a change in variance
# between the two halves of the record (Levene test centred on the median,
# i.e. the Brown-Forsythe test) and serial correlation
# (lag-1 autocorrelation and Ljung-Box test over five lags).
srows = []
for sid in ORDER:
    ys = YS[sid]
    am7 = [pd.Series(a).rolling(7).mean().min() for a in ys.Y]
    for name, x in [('Annual mean', ys.S / ys.C), ('Annual maximum', ys.AM), ('Annual 7-day minimum', am7)]:
        srows.append({'GRDC_No': sid, 'River': META[sid]['river'], 'Series': name, 'N': ys.N,
                      **trend_tests(np.asarray(x, float), ys.years)})
T_STAT = pd.DataFrame(srows)
T_STAT.to_csv(f'{OUT}/supplementary/TableS3_stationarity_persistence.csv', index=False)
log('\nStationarity / persistence'); log(T_STAT.round(3).to_string(index=False))

# ── Detrended records ────────────────────────────────────────────────────────
# For the sensitivity run, the daily flows of each year are multiplied by a
# factor that removes the Sen-slope trend of the annual mean flow while keeping
# the long-term mean unchanged.
YS_DT, DT_INFO = {}, []
for sid in REF_IDS:
    ys = YS[sid]
    ann = ys.S / ys.C
    b, a = sen_slope(ann)
    t = np.arange(ys.N)
    trend = a + b * t
    tmid = a + b * (ys.N - 1) / 2
    fac = np.where(trend > 0, tmid / trend, 1.0)
    YS_DT[sid] = YearSet([arr * f for arr, f in zip(ys.Y, fac)], ys.years)
    DT_INFO.append({'GRDC_No': sid, 'River': META[sid]['river'],
                    'Sen_slope_pct_per_decade': 10 * b / ann.mean() * 100,
                    'factor_min': fac.min(), 'factor_max': fac.max()})

# ── Resampling ───────────────────────────────────────────────────────────────
# For every reference station, every scheme and every record length n, B
# synthetic n-year records are built and all statistics are computed.
# Variants: 'main' (all valid days), 'nozero' (zero-flow days removed),
# 'detrended' (trend removed from the annual means).
SCHEMES_MAIN = ['sub', 'iid'] + [f'cbb{L}' for L in BLOCK_LENGTHS] + ['win']
RUNS = [('main', s) for s in SCHEMES_MAIN] + [('nozero', 'sub'), ('nozero', PRIMARY), ('detrended', PRIMARY)]
EST = {}
rng = np.random.default_rng(SEED)
log(f'\nResampling ({B} replicates per record length) ...')
for sid in REF_IDS:
    t0 = time.time()
    for var, sch in RUNS:
        ys = {'main': YS, 'nozero': YS_NZ, 'detrended': YS_DT}[var][sid]
        EST[(var, sch, sid)] = run_scheme(ys, sch, rng)
    log(f"  {META[sid]['river']:12s} done ({time.time() - t0:.0f} s)")

REFV = {'main': REF, 'nozero': REF_NZ, 'detrended': {s: YS_DT[s].reference() for s in REF_IDS}}


def get_err(var, sch, sid, stat, j, measure):
    """Errors of one station, scheme, statistic and record length."""
    est = EST[(var, sch, sid)][SI[stat], j, :]
    ref = REFV[var][sid][SI[stat]]
    return errors(est, ref, REFV[var][sid][SI['q_mean']], stat, measure)


# ── Error summaries ──────────────────────────────────────────────────────────
# For each station, scheme, statistic, record length and error measure: RMSE,
# mean bias with 95% interval, median and error percentiles.
SUMROWS = []
RMSE = {}   # (var, scheme, measure, sid, stat) -> array over SAMPLE_LENGTHS (pct units)
for (var, sch, sid) in EST:
    for stat in STAT_KEYS:
        for meas in MEASURES:
            arr = np.full(len(SAMPLE_LENGTHS), np.nan)
            for j, n in enumerate(SAMPLE_LENGTHS):
                e = get_err(var, sch, sid, stat, j, meas)
                sm = summarise(e, meas)
                if sm is None:
                    continue
                arr[j] = sm['RMSE']
                SUMROWS.append({'variant': var, 'scheme': sch, 'GRDC_No': sid,
                                'River': META[sid]['river'], 'stat': STAT_TXT[stat], 'n': n,
                                'measure': meas, **sm})
            RMSE[(var, sch, meas, sid, stat)] = arr
# FPC-corrected subsampling
for sid in REF_IDS:
    N = NYR[sid]
    fpc = np.array([np.sqrt((N - n) / (N - 1)) if n < N else np.nan for n in SAMPLE_LENGTHS])
    for stat in STAT_KEYS:
        for meas in MEASURES:
            RMSE[('main', 'sub_fpc', meas, sid, stat)] = RMSE[('main', 'sub', meas, sid, stat)] / fpc
T_SUM = pd.DataFrame(SUMROWS)
T_SUM.to_csv(f'{OUT}/supplementary/TableS4_error_summary_all.csv', index=False)

# Analytical curves for the mean
THEORY = {}
for sid in REF_IDS:
    ann = YS[sid].S / YS[sid].C
    cv0 = ann.std(ddof=0) / ann.mean()
    N = NYR[sid]
    ns = np.array(SAMPLE_LENGTHS, float)
    THEORY[sid] = {'iid': 100 * cv0 / np.sqrt(ns),
                   'sub': np.where(ns < N, 100 * cv0 * np.sqrt(np.clip(N - ns, 0, None) / (ns * (N - 1))), np.nan),
                   'cv0': cv0, 'N': N}

# ── Power laws for each station ──────────────────────────────────────────────
FIT_SCHEMES = SCHEMES_MAIN + ['sub_fpc']
frows = []
FITS = {}
for sch in FIT_SCHEMES:
    for sid in REF_IDS:
        for stat in STAT_KEYS:
            for meas in ['rel', 'log', 'norm']:
                f = fit_pl(SAMPLE_LENGTHS, RMSE[('main', sch, meas, sid, stat)])
                FITS[(sch, meas, sid, stat)] = f
                if f:
                    frows.append({'scheme': sch, 'measure': meas, 'GRDC_No': sid,
                                  'River': META[sid]['river'], 'stat': STAT_TXT[stat], **f})
for sid in REF_IDS:
    for key in ['iid', 'sub']:
        f = fit_pl(SAMPLE_LENGTHS, THEORY[sid][key])
        if f:
            frows.append({'scheme': f'theory_{key}', 'measure': 'rel', 'GRDC_No': sid,
                          'River': META[sid]['river'], 'stat': 'Qmean', **f})
            FITS[(f'theory_{key}', 'rel', sid, 'q_mean')] = f
for var in ['nozero', 'detrended']:
    for sch in ([PRIMARY, 'sub'] if var == 'nozero' else [PRIMARY]):
        for sid in REF_IDS:
            for stat in STAT_KEYS:
                for meas in ['rel', 'log']:
                    f = fit_pl(SAMPLE_LENGTHS, RMSE[(var, sch, meas, sid, stat)])
                    FITS[(f'{var}:{sch}', meas, sid, stat)] = f
                    if f:
                        frows.append({'scheme': f'{var}:{sch}', 'measure': meas, 'GRDC_No': sid,
                                      'River': META[sid]['river'], 'stat': STAT_TXT[stat], **f})
T_FIT = pd.DataFrame(frows)
T_FIT.to_csv(f'{OUT}/supplementary/TableS5_powerlaw_fits_single_station.csv', index=False)


# ── Pooled power laws ────────────────────────────────────────────────────────
# The mean and the maximum RMSE over the reference stations at each n are
# fitted. Confidence intervals of the pooled parameters are obtained by
# resampling the stations with replacement (catchment bootstrap).
def pooled_curves(sch, meas, stat, ids, var='main'):
    """Mean and maximum RMSE across the given stations at each n. Stations for
    which the statistic is undefined (e.g. a relative error when the reference
    value is zero) are left out; a record length is used only when all
    remaining stations have a value."""
    M = np.array([RMSE[(var, sch, meas, s, stat)] for s in ids])
    keep = np.isfinite(M).sum(axis=1) >= 4
    M = M[keep] if keep.any() else M
    ok = np.all(np.isfinite(M), axis=0)
    mean = np.where(ok, M.mean(axis=0), np.nan)
    mx = np.where(ok, M.max(axis=0), np.nan)
    return mean, mx, M


prow = []
POOLED = {}
rng_p = np.random.default_rng(SEED + 1)
for sch in FIT_SCHEMES:
    for stat in STAT_KEYS:
        for meas in ['rel', 'log', 'norm']:
            mean, mx, M = pooled_curves(sch, meas, stat, REF_IDS)
            fm, fu = fit_pl(SAMPLE_LENGTHS, mean), fit_pl(SAMPLE_LENGTHS, mx)
            POOLED[(sch, meas, stat)] = (fm, fu, mean, mx)
            if fm is None:
                continue
            row = {'scheme': sch, 'measure': meas, 'stat': STAT_TXT[stat],
                   **{f'mean_{k}': v for k, v in fm.items()},
                   **({f'upper_{k}': v for k, v in fu.items()} if fu else {})}
            if sch in (PRIMARY, 'sub', 'iid') and len(REF_IDS) >= 3:
                bb, aa = [], []
                ok = np.all(np.isfinite(M), axis=0)
                for _ in range(N_STATION_BOOT):
                    pick = rng_p.integers(0, len(M), len(M))
                    f = fit_pl(np.array(SAMPLE_LENGTHS)[ok], M[pick][:, ok].mean(axis=0))
                    if f:
                        aa.append(f['alpha'])
                        bb.append(f['beta'])
                if aa:
                    row.update({'mean_alpha_cb_lo': np.percentile(aa, 2.5), 'mean_alpha_cb_hi': np.percentile(aa, 97.5),
                                'mean_beta_cb_lo': np.percentile(bb, 2.5), 'mean_beta_cb_hi': np.percentile(bb, 97.5)})
            for n in [10, N_TARGET]:
                row[f'mean_pred_n{n}'] = pl_at(fm, n)
                row[f'upper_pred_n{n}'] = pl_at(fu, n)
            prow.append(row)
T_POOL = pd.DataFrame(prow)
T_POOL.to_csv(f'{OUT}/Table3a_pooled_powerlaw.csv', index=False)
log('\nPooled power laws (rel; log for Q90)')
_show = T_POOL[((T_POOL.measure == 'rel') & (T_POOL.stat != 'Q90')) | ((T_POOL.measure == 'log') & (T_POOL.stat == 'Q90'))]
log(_show[['scheme', 'stat', 'measure', 'mean_alpha', 'mean_beta', 'mean_beta_lo', 'mean_beta_hi',
           'mean_R2', 'upper_alpha', 'upper_beta']].round(3).to_string(index=False))

# ── Errors at n = 10 years for all statistics ───────────────────────────────
t2 = []
for sid in REF_IDS:
    for stat in STAT_KEYS:
        j = NI[10]
        rec = {'GRDC_No': sid, 'River': META[sid]['river'], 'stat': STAT_TXT[stat]}
        for meas in ['rel', 'log', 'norm']:
            sm = summarise(get_err('main', PRIMARY, sid, stat, j, meas), meas)
            if sm:
                for k in ['RMSE', 'mean_bias', 'mean_bias_lo', 'mean_bias_hi', 'median', 'P5', 'P95']:
                    rec[f'{meas}_{k}'] = sm[k]
        sm = summarise(get_err('main', 'sub', sid, stat, j, 'rel'), 'rel')
        rec['sub_rel_RMSE'] = sm['RMSE'] if sm else np.nan
        sm = summarise(get_err('main', 'iid', sid, stat, j, 'rel'), 'rel')
        rec['iid_rel_RMSE'] = sm['RMSE'] if sm else np.nan
        sm = summarise(get_err('main', 'win', sid, stat, j, 'rel'), 'rel')
        rec['win_rel_RMSE'] = sm['RMSE'] if sm else np.nan
        t2.append(rec)
T2 = pd.DataFrame(t2)
T2.to_csv(f'{OUT}/Table2_n10_primary.csv', index=False)
log(f'\nTable 2 (scheme {PRIMARY}, n = 10)')
log(T2[['River', 'stat', 'rel_RMSE', 'rel_mean_bias', 'rel_median', 'rel_P5', 'rel_P95', 'log_RMSE',
        'sub_rel_RMSE', 'iid_rel_RMSE', 'win_rel_RMSE']].round(2).to_string(index=False))

# ── Leave-one-catchment-out cross-validation ────────────────────────────────
# Each reference station is removed in turn, the models are fitted to the
# remaining stations and the RMSE curve of the removed station is predicted.
#   M1  pooled mean power law
#   M2  pooled upper (maximum) power law; reported as coverage of the observed curve
#   M3  scaled model RMSE = kappa * S * n^(-beta), where S is the variability
#       of the annual values of the statistic at the removed station
#       (M3b: S estimated from short contiguous windows of that station only)
#   M4  RMSE curve of the most similar station (nearest hydrological donor)
SCALE = {}   # (sid, stat, measure) -> S
for sid in ORDER:
    for stat in STAT_KEYS:
        for meas in ['rel', 'log', 'norm']:
            SCALE[(sid, stat, meas)] = scale_param(ANNUAL[sid][:, SI[stat]], stat, meas, QBAR[sid])


def fit_scaled(ids, sch, meas, stat):
    """Fit kappa and beta of the scaled model to the given stations."""
    xs, ns, ys_ = [], [], []
    for s in ids:
        S = SCALE[(s, stat, meas)]
        r = RMSE[('main', sch, meas, s, stat)]
        if meas == 'log':
            r = np.log1p(r / 100.0)          # back to log units
        for n, v in zip(SAMPLE_LENGTHS, r):
            if np.isfinite(v) and v > 0 and np.isfinite(S) and S > 0:
                xs.append(S); ns.append(n); ys_.append(v)
    if len(ys_) < 6:
        return None
    xs, ns, ys_ = map(np.asarray, (xs, ns, ys_))
    fun = lambda X, k, b: k * X[0] * 100.0 * np.power(X[1], -b) if meas != 'log' else k * X[0] * np.power(X[1], -b)
    try:
        popt, _ = curve_fit(fun, np.vstack([xs, ns]), ys_, p0=[1.0, 0.5], maxfev=20000)
    except Exception:
        return None
    return {'kappa': popt[0], 'beta': popt[1], 'fun': fun}


def pred_scaled(fs, S, meas):
    """RMSE curve predicted by the scaled model for scale parameter S."""
    if fs is None or not np.isfinite(S):
        return np.full(len(SAMPLE_LENGTHS), np.nan)
    v = fs['fun'](np.vstack([np.full(len(SAMPLE_LENGTHS), S), np.array(SAMPLE_LENGTHS, float)]),
                  fs['kappa'], fs['beta'])
    return 100 * np.expm1(v) if meas == 'log' else v


def score(pred, obs):
    """RMSE (percentage points), mean absolute percentage error and mean bias
    of a predicted RMSE curve against the observed curve."""
    ok = np.isfinite(pred) & np.isfinite(obs) & (obs > 0)
    if ok.sum() < 3:
        return np.nan, np.nan, np.nan
    e = pred[ok] - obs[ok]
    return np.sqrt(np.mean(e ** 2)), np.mean(np.abs(e) / obs[ok]) * 100, np.mean(e / obs[ok]) * 100


lrows, LOCO_PTS = [], []
LOCO_SCHEMES = [PRIMARY, 'iid', 'sub']
if len(REF_IDS) >= 4:
    for sch in LOCO_SCHEMES:
        for stat in STAT_KEYS:
            for meas in (['rel', 'log'] if stat in ('q90', 'q10', 'q50') else ['rel']):
                for out in REF_IDS:
                    train = [s for s in REF_IDS if s != out]
                    obs = RMSE[('main', sch, meas, out, stat)]
                    if not np.any(np.isfinite(obs)):
                        continue
                    mean, mx, _ = pooled_curves(sch, meas, stat, train)
                    fm, fu = fit_pl(SAMPLE_LENGTHS, mean), fit_pl(SAMPLE_LENGTHS, mx)
                    ns = np.array(SAMPLE_LENGTHS, float)
                    p1 = pl_at(fm, ns) if fm else np.full(len(ns), np.nan)
                    p2 = pl_at(fu, ns) if fu else np.full(len(ns), np.nan)
                    fs = fit_scaled(train, sch, meas, stat)
                    p3 = pred_scaled(fs, SCALE[(out, stat, meas)], meas)
                    don = nearest_donor(SIG, out, train)
                    p4 = RMSE[('main', sch, meas, don.index[0], stat)]
                    # scale parameter estimated from short contiguous windows of the held-out record
                    p3b_scores = []
                    ann = ANNUAL[out][:, SI[stat]]
                    for s0 in range(0, NYR[out] - N_TARGET + 1):
                        Sw = scale_param(ann[s0:s0 + N_TARGET], stat, meas, np.mean(YS[out].S[s0:s0 + N_TARGET] / YS[out].C[s0:s0 + N_TARGET]))
                        p3b_scores.append(score(pred_scaled(fs, Sw, meas), obs))
                    p3b = np.array(p3b_scores, float) if p3b_scores else np.full((1, 3), np.nan)
                    rec = {'scheme': sch, 'measure': meas, 'stat': STAT_TXT[stat], 'held_out': META[out]['river'],
                           'donor': META[don.index[0]]['river'], 'donor_distance': don.iloc[0]}
                    for name, p in [('M1_pooled_mean', p1), ('M3_scaled_fullS', p3), ('M4_nearest_donor', p4)]:
                        r_, mape, bias = score(p, obs)
                        rec[f'{name}_RMSEpp'], rec[f'{name}_MAPE'], rec[f'{name}_bias_pct'] = r_, mape, bias
                    rec['M3b_scaled_shortS_RMSEpp_median'] = np.nanmedian(p3b[:, 0])
                    rec['M3b_scaled_shortS_RMSEpp_P90'] = np.nanpercentile(p3b[:, 0], 90)
                    rec['M3b_scaled_shortS_MAPE_median'] = np.nanmedian(p3b[:, 1])
                    ok = np.isfinite(p2) & np.isfinite(obs)
                    rec['M2_upper_coverage_pct'] = 100 * np.mean(obs[ok] <= p2[ok]) if ok.any() else np.nan
                    rec['kappa'] = fs['kappa'] if fs else np.nan
                    rec['beta_scaled'] = fs['beta'] if fs else np.nan
                    lrows.append(rec)
                    if sch == PRIMARY:
                        for j, n in enumerate(SAMPLE_LENGTHS):
                            LOCO_PTS.append({'stat': stat, 'measure': meas, 'held_out': out, 'n': n,
                                             'obs': obs[j], 'M1': p1[j], 'M2': p2[j], 'M3': p3[j], 'M4': p4[j]})
T_LOCO = pd.DataFrame(lrows)
T_LOCO.to_csv(f'{OUT}/supplementary/TableS6_LOCO_by_catchment.csv', index=False)
LOCO_PTS = pd.DataFrame(LOCO_PTS)
LOCO_PTS.to_csv(f'{OUT}/supplementary/TableS6b_LOCO_points.csv', index=False)
if len(T_LOCO):
    agg_cols = [c for c in T_LOCO.columns if c.endswith(('RMSEpp', 'MAPE', 'coverage_pct', 'RMSEpp_median',
                                                         'RMSEpp_P90', 'MAPE_median', 'bias_pct'))]
    T_LOCO_SUM = T_LOCO.groupby(['scheme', 'measure', 'stat'])[agg_cols].agg(['mean', 'max']).reset_index()
    T_LOCO_SUM.columns = ['_'.join([c for c in col if c]) for col in T_LOCO_SUM.columns]
    T_LOCO_SUM.to_csv(f'{OUT}/Table3b_LOCO_summary.csv', index=False)
    log('\nLOCO summary (mean over held-out catchments)')
    cols = ['scheme', 'measure', 'stat'] + [c for c in T_LOCO_SUM.columns if c.endswith('_mean') and
                                            ('MAPE' in c or 'coverage' in c)]
    log(T_LOCO_SUM[cols].round(1).to_string(index=False))

# ── Application to the short-record station ─────────────────────────────────
# The station most similar to the target is identified from standardised
# hydrological signatures. Expected errors at the target record length are
# given by each model, and the donor's error percentiles are converted into an
# interval for the long-term value of each statistic.
if HAS_TARGET:
    DON = nearest_donor(SIG, TARGET_ID, REF_IDS)
    T_DON = pd.DataFrame({'River': [META[s]['river'] for s in DON.index], 'distance': DON.values})
    T_DON.to_csv(f'{OUT}/supplementary/TableS7_donor_similarity.csv', index=False)
    log('\nHydrological similarity to target (standardised signature distance)')
    log(T_DON.round(3).to_string(index=False))
    donor = DON.index[0]
    rng_t = np.random.default_rng(SEED + 2)
    arows = []
    obs_t = REF[TARGET_ID]
    for stat in STAT_KEYS:
        for meas in (['rel', 'log', 'norm'] if stat == 'q90' else ['rel', 'log'] if stat in ('q10', 'q50') else ['rel']):
            fm, fu, _, _ = POOLED[(PRIMARY, meas, stat)]
            fs = fit_scaled(REF_IDS, PRIMARY, meas, stat)
            ann = ANNUAL[TARGET_ID][:, SI[stat]]
            S_t = SCALE[(TARGET_ID, stat, meas)]
            Sb = []
            for _ in range(N_TARGET_BOOT):
                pick = rng_t.integers(0, len(ann), len(ann))
                Sb.append(scale_param(ann[pick], stat, meas, QBAR[TARGET_ID]))
            Sb = np.array(Sb, float)
            pS = pred_scaled(fs, S_t, meas)
            pS_lo = pred_scaled(fs, np.nanpercentile(Sb, 5), meas)
            pS_hi = pred_scaled(fs, np.nanpercentile(Sb, 95), meas)
            dfit = fit_pl(SAMPLE_LENGTHS, RMSE[('main', PRIMARY, meas, donor, stat)])
            rec = {'stat': STAT_TXT[stat], 'measure': meas, 'N_target': N_TARGET,
                   'target_value': obs_t[SI[stat]], 'S_target': S_t,
                   'S_target_P5': np.nanpercentile(Sb, 5), 'S_target_P95': np.nanpercentile(Sb, 95),
                   'pooled_mean_RMSE': pl_at(fm, N_TARGET), 'pooled_upper_RMSE': pl_at(fu, N_TARGET),
                   'donor': META[donor]['river'], 'donor_RMSE': pl_at(dfit, N_TARGET),
                   'scaled_RMSE': np.interp(N_TARGET, SAMPLE_LENGTHS, pS),
                   'scaled_RMSE_P5S': np.interp(N_TARGET, SAMPLE_LENGTHS, pS_lo),
                   'scaled_RMSE_P95S': np.interp(N_TARGET, SAMPLE_LENGTHS, pS_hi)}
            if meas == 'rel' and stat != 'cv':
                # interval for the long-term value implied by donor error percentiles
                jlo = max(j for j, n in enumerate(SAMPLE_LENGTHS) if n <= N_TARGET)
                e = get_err('main', PRIMARY, donor, stat, jlo, 'rel')
                e = e[np.isfinite(e)]
                if len(e):
                    p5, p95 = np.percentile(e, [5, 95])
                    rec['donor_err_P5'], rec['donor_err_P95'] = p5, p95
                    rec['longterm_lo'] = obs_t[SI[stat]] / (1 + p95 / 100)
                    rec['longterm_hi'] = obs_t[SI[stat]] / (1 + p5 / 100)
            arows.append(rec)
    T_APP = pd.DataFrame(arows)
    T_APP.to_csv(f'{OUT}/Table4_target_application.csv', index=False)
    log(f"\nApplication to {META[TARGET_ID]['river']} (N = {N_TARGET} complete years, donor = {META[donor]['river']})")
    log(T_APP.round(3).to_string(index=False))

# ── Sensitivity analyses ─────────────────────────────────────────────────────
# Low-flow index with and without zero-flow days; detrended records; block
# length; relation between errors and catchment descriptors; bias table.
zrows = []
for sid in REF_IDS:
    for sch in [PRIMARY, 'sub']:
        rec = {'River': META[sid]['river'], 'scheme': sch,
               'zero_days': int((DAILY[sid] == 0).sum()),
               'Q90_with_zeros': REF[sid][SI['q90']], 'Q90_without_zeros': REF_NZ[sid][SI['q90']]}
        for var in ['main', 'nozero']:
            for meas in ['rel', 'log', 'norm']:
                r = RMSE[(var, sch, meas, sid, 'q90')]
                rec[f'{var}_{meas}_RMSE_n10'] = r[NI[10]]
                rec[f'{var}_{meas}_RMSE_n20'] = r[NI[20]]
            f = FITS.get((sch if var == 'main' else f'nozero:{sch}', 'log', sid, 'q90'))
            rec[f'{var}_log_beta'] = f['beta'] if f else np.nan
        zrows.append(rec)
T_ZS = pd.DataFrame(zrows)
T_ZS.to_csv(f'{OUT}/supplementary/TableS8_zero_flow_sensitivity_Q90.csv', index=False)
log('\nQ90 with and without zero-flow days'); log(T_ZS.round(3).to_string(index=False))

drows = []
for sid in REF_IDS:
    info = [d for d in DT_INFO if d['River'] == META[sid]['river']][0]
    for stat in ['q_mean', 'ann_max', 'q90', 'q100']:
        meas = 'log' if stat == 'q90' else 'rel'
        f0 = FITS.get((PRIMARY, meas, sid, stat))
        f1 = FITS.get((f'detrended:{PRIMARY}', meas, sid, stat))
        drows.append({'River': META[sid]['river'], 'stat': STAT_TXT[stat], 'measure': meas,
                      'Sen_slope_pct_per_decade': info['Sen_slope_pct_per_decade'],
                      'RMSE10_original': RMSE[('main', PRIMARY, meas, sid, stat)][NI[10]],
                      'RMSE10_detrended': RMSE[('detrended', PRIMARY, meas, sid, stat)][NI[10]],
                      'beta_original': f0['beta'] if f0 else np.nan,
                      'beta_detrended': f1['beta'] if f1 else np.nan})
        for L in BLOCK_LENGTHS:
            drows[-1][f'RMSE10_cbb{L}'] = RMSE[('main', f'cbb{L}', meas, sid, stat)][NI[10]]
        drows[-1]['RMSE10_iid'] = RMSE[('main', 'iid', meas, sid, stat)][NI[10]]
        drows[-1]['RMSE10_win'] = RMSE[('main', 'win', meas, sid, stat)][NI[10]]
T_DT = pd.DataFrame(drows)
T_DT.to_csv(f'{OUT}/supplementary/TableS9_detrending_blocklength_sensitivity.csv', index=False)
log('\nDetrending and block-length sensitivity'); log(T_DT.round(2).to_string(index=False))

# Drivers of between-catchment differences
crow = []
desc = {'Area_km2': [META[s]['area'] for s in REF_IDS],
        'CV_annual': [SIG.loc[s, 'CV_annual'] for s in REF_IDS],
        'CV_daily': [SIG.loc[s, 'CV_daily'] for s in REF_IDS],
        'BFI': [SIG.loc[s, 'BFI'] for s in REF_IDS],
        'zero_frac': [SIG.loc[s, 'zero_frac'] for s in REF_IDS],
        'r1_annual': [SIG.loc[s, 'r1_annual'] for s in REF_IDS],
        'Seasonality_R': [SIG.loc[s, 'Seasonality_R'] for s in REF_IDS],
        'Flashiness_RB': [SIG.loc[s, 'Flashiness_RB'] for s in REF_IDS]}
for stat in ['q_mean', 'ann_max', 'q90', 'q100']:
    meas = 'log' if stat == 'q90' else 'rel'
    y = [RMSE[('main', PRIMARY, meas, s, stat)][NI[10]] for s in REF_IDS]
    for k, x in desc.items():
        x = np.asarray(x, float)
        ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() >= 4 and np.std(x[ok]) > 0:
            rho, p = stats.spearmanr(x[ok], np.asarray(y)[ok])
            crow.append({'stat': STAT_TXT[stat], 'descriptor': k, 'spearman_rho': rho, 'p': p, 'n': int(ok.sum())})
    crow.append({'stat': STAT_TXT[stat], 'descriptor': 'S (own scale parameter)',
                 'spearman_rho': stats.spearmanr([SCALE[(s, stat, meas)] for s in REF_IDS], y)[0],
                 'p': stats.spearmanr([SCALE[(s, stat, meas)] for s in REF_IDS], y)[1], 'n': len(REF_IDS)})
T_DRV = pd.DataFrame(crow)
T_DRV.to_csv(f'{OUT}/supplementary/TableS10_descriptor_correlations.csv', index=False)
log('\nSpearman correlations of RMSE(n=10) with catchment descriptors'); log(T_DRV.round(3).to_string(index=False))

# Bias table for all statistics and record lengths
T_BIAS = T_SUM[(T_SUM.variant == 'main') & (T_SUM.measure == 'rel')][
    ['scheme', 'River', 'stat', 'n', 'mean_bias', 'mean_bias_lo', 'mean_bias_hi', 'median', 'P5', 'P95']]
T_BIAS.to_csv(f'{OUT}/supplementary/TableS11_bias_all.csv', index=False)


# =============================================================================
#  FIGURES
#  Main figures: Fig1-Fig5. Supplementary figures: FigS1-FigS8.
#  Reference stations are coloured by climate group; the short-record station
#  is shown in magenta.
# =============================================================================
def savefig(fig, name):
    """Save a figure at 600 dpi (RGB, white background) and show it."""
    fig.savefig(f'{OUT}/{name}', dpi=600, bbox_inches='tight', facecolor='white')
    try:  # store as RGB (no transparency channel)
        from PIL import Image
        im = Image.open(f'{OUT}/{name}')
        if im.mode != 'RGB':
            im.convert('RGB').save(f'{OUT}/{name}', dpi=(600, 600))
    except Exception:
        pass
    plt.show()
    plt.close(fig)
    log(f'  saved {name}')


def safe(fn):
    """Draw a figure; report an error without stopping the run."""
    try:
        fn()
    except Exception:
        log('Figure failed:')
        traceback.print_exc()


def lab(sid):
    """River name used in labels."""
    return META[sid]['river']


NS = np.array(SAMPLE_LENGTHS, float)
NF = np.linspace(5, 50, 200)


def _plain_log_labels(axis, subs=(1, 2, 5)):
    def fmt(v, p):
        if v <= 0:
            return ''
        m = v / 10 ** np.floor(np.log10(v) + 1e-9)
        return f'{v:g}' if any(abs(m - s_) < 1e-6 for s_ in subs) else ''
    axis.set_major_formatter(matplotlib.ticker.FuncFormatter(fmt))
    axis.set_minor_formatter(matplotlib.ticker.FuncFormatter(fmt))


def loglog_axes(ax):
    """Logarithmic axes with plain tick labels."""
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xticks([5, 10, 20, 50]); ax.set_xticklabels(['5', '10', '20', '50'])
    ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    _plain_log_labels(ax.yaxis)
    ax.grid(True, which='both', lw=0.3, alpha=0.35)


def group_handles(ls='-', marker='o', lw=1.0):
    h = [Line2D([], [], color=GROUP_COL[g], ls=ls, marker=marker, ms=3, lw=lw,
                label=f'{g}, {sum(GROUP[s] == g for s in REF_IDS)} catchments') for g in GROUPS_PRESENT]
    return h


def members(g):
    return [s for s in REF_IDS if GROUP[s] == g]


def grid(n, ncol, w=6.85, h=1.9):
    nrow = int(np.ceil(n / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(w, h * nrow))
    axs = np.atleast_1d(axs).ravel()
    for j in range(n, len(axs)):
        axs[j].set_visible(False)
    return fig, axs


# Fig 1: map, flow-duration curves, regimes
def fig1(use_cartopy=True):
    """Fig. 1: (a) station map with station numbers (see Table 1); (b) flow-
    duration curves normalised by mean flow; (c) mean monthly regimes."""
    fig = plt.figure(figsize=(6.85, 8.4))
    gs = gridspec.GridSpec(2, 2, height_ratios=[1.45, 1], hspace=0.32, wspace=0.3)
    lats = [META[s]['lat'] for s in ORDER if META[s].get('lat') is not None]
    lons = [META[s]['lon'] for s in ORDER if META[s].get('lon') is not None]
    ext = [min(lons) - 3, max(lons) + 3, min(lats) - 2.5, max(lats) + 2.5]
    try:
        if not use_cartopy:
            raise ImportError
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        ax = fig.add_subplot(gs[0, :], projection=ccrs.PlateCarree())
        ax.set_extent(ext, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='#f0efea', edgecolor='none')
        ax.add_feature(cfeature.OCEAN, facecolor='#d6eaf8')
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='#aaaaaa')
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#888888')
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = gl.ylabel_style = {'size': 7}
        tr = dict(transform=ccrs.PlateCarree())
    except Exception:
        ax = fig.add_subplot(gs[0, :])
        ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:]); ax.set_facecolor('#d6eaf8')
        ax.set_aspect('equal')
        ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')
        tr = {}
    texts = []
    for sid in ORDER:
        m = META[sid]
        if m.get('lat') is None:
            continue
        tgt = sid == TARGET_ID
        ax.plot(m['lon'], m['lat'], marker='*' if tgt else 'o', color=COLORS[sid], ms=14 if tgt else 6.5,
                mec='k', mew=0.5, zorder=5, **tr)
        texts.append(ax.text(m['lon'] + 0.35, m['lat'] + 0.3, str(NUMBER[sid]), fontsize=7,
                             fontweight='bold', zorder=6, **tr))
    try:
        from adjustText import adjust_text
        adjust_text(texts, ax=ax, expand=(1.3, 1.5),
                    arrowprops=dict(arrowstyle='-', color='k', lw=0.4))
    except Exception:
        pass
    hand = group_handles(ls='', marker='o') + [Line2D([], [], color=TARGET_COL, marker='*', ls='', ms=10,
                                                      mec='k', mew=0.5, label=f'{lab(TARGET_ID)} (short record)')]
    ax.legend(handles=hand, fontsize=7, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False)
    ax.set_title('(a) Gauging stations (numbers as in Table 1)', loc='left')
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])
    for sid in REF_IDS + ([TARGET_ID] if HAS_TARGET else []):
        tgt = sid == TARGET_ID
        kw = dict(color=COLORS[sid], ls='--' if tgt else '-', lw=1.8 if tgt else 0.7,
                  alpha=1 if tgt else 0.6, zorder=5 if tgt else 2)
        x = np.sort(DAILY[sid].dropna().values)[::-1]
        p = np.arange(1, len(x) + 1) / (len(x) + 1)
        y = x / QBAR[sid]
        ax2.semilogy(p * 100, np.where(y > 0, y, np.nan), **kw)
        mq = DAILY[sid].groupby(DAILY[sid].index.month).mean()
        mq = mq.reindex([10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9])
        ax3.plot(range(12), mq.values / QBAR[sid], **kw)
    ax2.set_xlabel('Exceedance probability (%)'); ax2.set_ylabel(r'$Q/\bar{Q}$ (–)')
    ax2.set_ylim(bottom=1e-3)
    ax2.set_title('(b) Flow-duration curves', loc='left'); ax2.grid(True, which='both', lw=0.3, alpha=0.35)
    ax3.set_xticks(range(12)); ax3.set_xticklabels(list('ONDJFMAMJJAS'))
    ax3.set_ylabel(r'Monthly mean / $\bar{Q}$ (–)'); ax3.set_title('(c) Mean monthly regimes', loc='left')
    ax3.grid(True, lw=0.3, alpha=0.35)
    try:
        fig.canvas.draw()
    except Exception:
        plt.close(fig)
        if use_cartopy:
            log('  map background unavailable - drawing plain map')
            return fig1(use_cartopy=False)
        raise
    savefig(fig, 'Fig1_study_area_regimes.png')


# Fig 2: resampling schemes for the mean, by climate group
def fig2():
    """Fig. 2: RMSE of mean discharge versus record length for the resampling
    schemes (group means) and the analytical curves for independent years and
    for sampling without replacement."""
    panels = GROUPS_PRESENT + ['All reference catchments']
    nc = 2 if len(panels) in (2, 4) else 3
    fig, axs = grid(len(panels), nc, w=6.85 if nc == 3 else 5.75, h=2.45)
    for i, g in enumerate(panels):
        ax = axs[i]
        ids = REF_IDS if g.startswith('All') else members(g)
        for sch in ['sub', 'iid', PRIMARY, 'win']:
            M = np.array([RMSE[('main', sch, 'rel', s, 'q_mean')] for s in ids])
            st = SCHEME_STYLE[sch]
            ax.plot(NS, np.nanmean(M, axis=0), ls=st['ls'], marker=st['marker'], color=st['color'], ms=3, lw=1.1)
        ax.plot(NS, np.nanmean([THEORY[s]['iid'] for s in ids], axis=0), color='k', lw=0.8, ls='-.')
        ax.plot(NS, np.nanmean([THEORY[s]['sub'] for s in ids], axis=0), color='k', lw=0.8, ls=':')
        loglog_axes(ax)
        bs = [FITS[('sub', 'rel', s, 'q_mean')]['beta'] for s in ids if FITS.get(('sub', 'rel', s, 'q_mean'))]
        bp = [FITS[(PRIMARY, 'rel', s, 'q_mean')]['beta'] for s in ids if FITS.get((PRIMARY, 'rel', s, 'q_mean'))]
        ax.text(0.04, 0.05, f"mean $\\beta$: subsampling {np.mean(bs):.2f}\n"
                            f"mean $\\beta$: block bootstrap {np.mean(bp):.2f}", transform=ax.transAxes, fontsize=7)
        ax.set_title(f"({chr(97 + i)}) {g} ({len(ids)})", loc='left', fontsize=8.5)
        ax.set_xlabel('Record length n (years)', fontsize=8)
        if i % nc == 0:
            ax.set_ylabel(r'RMSE of $\bar{Q}$ (%)')
    hand = [Line2D([], [], **SCHEME_STYLE[s], ms=3, lw=1.1, label=SCHEME_LABEL[s]) for s in ['sub', 'iid', PRIMARY, 'win']]
    hand += [Line2D([], [], color='k', ls='-.', lw=0.8, label=r'Analytical, independent years: $100\,CV_a\,n^{-1/2}$'),
             Line2D([], [], color='k', ls=':', lw=0.8, label='Analytical, without replacement (finite-population correction)')]
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    fig.legend(handles=hand, loc='lower center', ncol=2, fontsize=7, frameon=False, bbox_to_anchor=(0.5, 0.0))
    savefig(fig, 'Fig2_resampling_schemes_mean.png')


# Fig 3: primary-scheme RMSE decay for all statistics
def fig3():
    """Fig. 3: RMSE versus record length for all statistics (primary scheme)."""
    fig, axs = plt.subplots(2, 4, figsize=(6.85, 4.4), sharex=True)
    axs = axs.ravel()
    for i, stat in enumerate(STAT_KEYS):
        ax = axs[i]
        meas = 'log' if stat == 'q90' else 'rel'
        for sid in REF_IDS:
            ax.plot(NS, RMSE[('main', PRIMARY, meas, sid, stat)], '-', color=COLORS[sid], lw=0.7, alpha=0.65)
        fm, fu, _, _ = POOLED[(PRIMARY, meas, stat)]
        if fm:
            ax.plot(NF, pl_at(fm, NF), color='k', lw=1.7)
            ax.text(0.97, 0.96, f"{fm['alpha']:.1f} $n^{{-{fm['beta']:.2f}}}$", transform=ax.transAxes,
                    ha='right', va='top', fontsize=7)
        if fu:
            ax.plot(NF, pl_at(fu, NF), color='k', lw=1.0, ls='--')
        ax.axvline(N_TARGET, color=TARGET_COL, ls=':', lw=1)
        loglog_axes(ax)
        ttl = STAT_TEX[stat] + (' (log-ratio)' if meas == 'log' else '')
        ax.set_title(f'({chr(97 + i)}) {ttl}', loc='left', fontsize=8.5)
        if i % 4 == 0:
            ax.set_ylabel('RMSE (%)')
        if i >= 3:
            ax.set_xlabel('n (years)')
    axs[3].xaxis.set_tick_params(labelbottom=True)
    axl = axs[7]
    axl.axis('off')
    hand = group_handles(marker=None)
    hand += [Line2D([], [], color='k', lw=1.7, label='Pooled mean'),
             Line2D([], [], color='k', ls='--', lw=1.0, label='Pooled upper (maximum)'),
             Line2D([], [], color=TARGET_COL, ls=':', lw=1, label=f'n = {N_TARGET}')]
    axl.legend(handles=hand, loc='center', fontsize=7, frameon=False)
    fig.tight_layout()
    savefig(fig, 'Fig3_rmse_decay_primary.png')


# Fig 4: low-flow error measures and zero-flow sensitivity
def fig4():
    """Fig. 4: low-flow index - relative versus log-ratio errors, dependence of
    each error measure on the low-flow ratio Q90/Qmean, effect of zero-flow
    days, and errors normalised by mean flow."""
    fig = plt.figure(figsize=(6.85, 5.6))
    gs = gridspec.GridSpec(2, 2, hspace=0.5, wspace=0.38)
    ax = fig.add_subplot(gs[0, 0])
    allv = []
    for sid in REF_IDS:
        x = RMSE[('main', PRIMARY, 'rel', sid, 'q90')]
        y = RMSE[('main', PRIMARY, 'log', sid, 'q90')]
        ok = np.isfinite(x) & np.isfinite(y)
        ax.scatter(x[ok], y[ok], s=5 + NS[ok] * 0.5, color=COLORS[sid], alpha=0.7, edgecolor='none')
        allv += list(x[ok]) + list(y[ok])
    lim = [max(1, min(allv) * 0.8), max(allv) * 1.2] if allv else [1, 100]
    ax.plot(lim, lim, 'k--', lw=0.7)
    ax.set_xscale('log'); ax.set_yscale('log')
    _plain_log_labels(ax.xaxis); _plain_log_labels(ax.yaxis)
    ax.set_xlabel('RMSE, relative error (%)'); ax.set_ylabel('RMSE, log-ratio (% equiv.)')
    ax.set_title(r'(a) $Q_{90}$: relative vs log-ratio RMSE', loc='left', fontsize=8.5)
    ax.grid(True, which='both', lw=0.3, alpha=0.35)
    ax.legend(handles=group_handles(ls=''), fontsize=7, loc='upper left', framealpha=0.7)
    ax = fig.add_subplot(gs[0, 1])
    j = NI[10]
    for sid in REF_IDS:
        r = SIG.loc[sid, 'Q90_over_Qbar']
        if not r > 0:
            continue
        xr = r
        for meas, mk, fc in [('rel', 'o', COLORS[sid]), ('log', 's', 'none'), ('norm', '^', 'none')]:
            v = RMSE[('main', PRIMARY, meas, sid, 'q90')][j]
            if np.isfinite(v):
                ax.scatter(xr, v, marker=mk, s=16, facecolor=fc, edgecolor=COLORS[sid], lw=0.7)
    ax.set_xscale('log'); ax.set_yscale('log')
    _plain_log_labels(ax.yaxis)
    ax.set_xlabel(r'$Q_{90}/\bar{Q}$ (–)'); ax.set_ylabel('RMSE at n = 10 (%)')
    ax.set_title(r'(b) $Q_{90}$ error measures vs low-flow ratio', loc='left', fontsize=8.5)
    ax.grid(True, which='both', lw=0.3, alpha=0.35)
    ax.legend(handles=[Line2D([], [], marker='o', ls='', color='gray', ms=4, label='relative'),
                       Line2D([], [], marker='s', ls='', mfc='none', mec='gray', ms=4, label='log-ratio'),
                       Line2D([], [], marker='^', ls='', mfc='none', mec='gray', ms=4, label=r'normalised by $\bar{Q}$')],
              fontsize=7, loc='upper right', framealpha=0.7)
    ax = fig.add_subplot(gs[1, 0])
    zids = [s for s in REF_IDS if (DAILY[s] == 0).sum() > 0]
    if zids:
        for sid in zids:
            ax.plot(NS, RMSE[('main', PRIMARY, 'log', sid, 'q90')], '-o', color=COLORS[sid], ms=2.5, lw=1)
            ax.plot(NS, RMSE[('nozero', PRIMARY, 'log', sid, 'q90')], ':', color=COLORS[sid], lw=1.3)
        loglog_axes(ax)
    else:
        ax.text(0.5, 0.5, 'No zero-flow days', ha='center', transform=ax.transAxes)
    ax.set_xlabel('n (years)'); ax.set_ylabel('RMSE, log-ratio (% equiv.)')
    ax.set_title(r'(c) $Q_{90}$ with (solid) and without (dotted) zero days', loc='left', fontsize=8)
    ax = fig.add_subplot(gs[1, 1])
    for sid in REF_IDS:
        ax.plot(NS, RMSE[('main', PRIMARY, 'norm', sid, 'q90')], '-', color=COLORS[sid], lw=0.8, alpha=0.7)
    loglog_axes(ax)
    ax.set_xlabel('n (years)'); ax.set_ylabel(r'RMSE / $\bar{Q}$ (%)')
    ax.set_title(r'(d) $Q_{90}$ error normalised by mean flow', loc='left', fontsize=8.5)
    savefig(fig, 'Fig4_lowflow_measures_zeros.png')


# Fig 5: exponents with CIs and leave-one-out validation
def fig5():
    """Fig. 5: power-law exponents of mean discharge with 95% intervals for each
    scheme, and leave-one-catchment-out predictions versus observations."""
    fig = plt.figure(figsize=(6.85, 3.7))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1.1, 1], wspace=0.3)
    ax = fig.add_subplot(gs[0])
    cols = [('theory_sub', 'Analytical, without repl.'), ('sub', 'Subsampling'), ('sub_fpc', 'Subsampling + FPC'),
            ('iid', 'Independent years'), (PRIMARY, f'Blocks, L = {PRIMARY[3:]} yr'), ('win', 'Contiguous windows')]
    rng_j = np.random.default_rng(7)
    for k_, (sch, _) in enumerate(cols):
        bs = []
        for sid in REF_IDS:
            f = FITS.get((sch, 'rel', sid, 'q_mean'))
            if not f:
                continue
            x = k_ + rng_j.uniform(-0.28, 0.28)
            ax.errorbar(x, f['beta'], yerr=[[max(f['beta'] - f['beta_lo'], 0)], [max(f['beta_hi'] - f['beta'], 0)]],
                        fmt='o', color=COLORS[sid], ms=2.5, lw=0.5, alpha=0.8, capsize=0)
            bs.append(f['beta'])
        if bs:
            ax.plot([k_ - 0.38, k_ + 0.38], [np.median(bs)] * 2, color='k', lw=1.6)
    ax.axhline(0.5, color='k', lw=0.8, ls='-.')
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([c[1] for c in cols], fontsize=7, rotation=30, ha='right')
    ax.set_ylim(0.25, 1.12)
    ax.set_yticks(np.arange(0.3, 1.01, 0.1))
    ax.set_ylabel(r'Exponent $\beta$ for $\bar{Q}$ (95% CI)')
    ax.set_title('(a) Fitted exponents by resampling scheme', loc='left', fontsize=8.5)
    ax.grid(True, axis='y', lw=0.3, alpha=0.35)
    ax.legend(handles=group_handles(ls='') + [Line2D([], [], color='k', lw=1.6, label='Median')],
              fontsize=7, loc='upper center', bbox_to_anchor=(0.5, -0.36), ncol=2, frameon=False)
    ax = fig.add_subplot(gs[1])
    if len(LOCO_PTS):
        mk = {'q_mean': 'o', 'ann_max': 's', 'q50': '^', 'q100': 'D'}
        d0 = LOCO_PTS[LOCO_PTS.measure == 'rel']
        for stat, m_ in mk.items():
            d = d0[d0.stat == stat]
            ax.scatter(d.obs, d.M1, marker=m_, facecolor='none', edgecolor='#7f7f7f', s=10, lw=0.5)
            ax.scatter(d.obs, d.M3, marker=m_, color='#d62728', s=8, alpha=0.6, edgecolor='none')
        vals = d0[['obs', 'M1', 'M3']].values
        vals = vals[np.isfinite(vals) & (vals > 0)]
        lo, hi = vals.min() * 0.8, vals.max() * 1.2
        ax.plot([lo, hi], [lo, hi], 'k-', lw=0.7)
        ax.plot([lo, hi], [lo * 1.25, hi * 1.25], 'k:', lw=0.5)
        ax.plot([lo, hi], [lo / 1.25, hi / 1.25], 'k:', lw=0.5)
        ax.set_xscale('log'); ax.set_yscale('log')
        _plain_log_labels(ax.xaxis); _plain_log_labels(ax.yaxis)
        hand = [Line2D([], [], marker=m_, ls='', color='k', mfc='none', ms=4, label=STAT_TEX[s]) for s, m_ in mk.items()]
        hand += [Line2D([], [], marker='o', ls='', mfc='none', mec='#7f7f7f', ms=4, label='Pooled power law'),
                 Line2D([], [], marker='o', ls='', color='#d62728', ms=4, label='Scaled model'),
                 Line2D([], [], color='k', ls=':', lw=0.5, label='±25%')]
        ax.legend(handles=hand, fontsize=7, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2,
                  frameon=False, columnspacing=1.0, handletextpad=0.3)
    ax.set_xlabel('Observed RMSE, held-out catchment (%)'); ax.set_ylabel('Predicted RMSE (%)')
    ax.set_title('(b) Leave-one-catchment-out validation', loc='left', fontsize=8.5)
    ax.grid(True, which='both', lw=0.3, alpha=0.35)
    savefig(fig, 'Fig5_exponents_LOCO.png')


# Supplementary figures
def figS1():
    """Fig. S1: daily discharge series; grey shading marks years not used."""
    fig, axs = grid(len(ORDER), 3, h=1.25)
    for ax, sid in zip(axs, ORDER):
        d = raw[sid][1]
        ax.fill_between(d.index, d.values, color=COLORS[sid], alpha=0.75, lw=0)
        unused = sorted(set(np.unique(water_year(d.index))) - set(YS[sid].years))
        for y in unused:
            ax.axvspan(pd.Timestamp(y - 1, WATER_YEAR_START, 1), pd.Timestamp(y, WATER_YEAR_START, 1),
                       color='gray', alpha=0.25, lw=0)
        ax.set_title(f"{lab(sid)} ({META[sid]['country']})", fontsize=7, loc='left', pad=1)
        ax.tick_params(labelsize=7)
        ax.xaxis.set_major_locator(matplotlib.dates.AutoDateLocator(minticks=3, maxticks=5))
        ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%Y'))
    fig.tight_layout(h_pad=0.4, w_pad=0.6)
    savefig(fig, 'supplementary/FigS1_timeseries.png')


def figS2():
    """Fig. S2: mean monthly discharge (water year October-September)."""
    fig, axs = grid(len(ORDER), 4, h=1.35)
    for ax, sid in zip(axs, ORDER):
        mq = DAILY[sid].groupby(DAILY[sid].index.month).mean().reindex([10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9])
        ax.bar(range(12), mq.values, color=COLORS[sid], alpha=0.85, width=0.75)
        ax.set_xticks(range(12)); ax.set_xticklabels(list('ONDJFMAMJJAS'), fontsize=7)
        ax.set_title(lab(sid), fontsize=7, pad=1)
        ax.tick_params(labelsize=7)
    fig.tight_layout(h_pad=0.3, w_pad=0.3)
    savefig(fig, 'supplementary/FigS2_monthly_regimes.png')


def figS3():
    """Fig. S3: distributions of relative errors at n = 10 (diamonds: means)."""
    fig, axs = plt.subplots(3, 1, figsize=(6.85, 6.6), sharex=True)
    for ax, stat in zip(axs, ['q_mean', 'ann_max', 'q100']):
        data = [get_err('main', PRIMARY, s, stat, NI[10], 'rel') for s in REF_IDS]
        data = [d[np.isfinite(d)] for d in data]
        vp = ax.violinplot(data, showmedians=True, showextrema=False, widths=0.8)
        for pc, s in zip(vp['bodies'], REF_IDS):
            pc.set_facecolor(COLORS[s]); pc.set_alpha(0.6)
        vp['cmedians'].set_color('k')
        ax.scatter(range(1, len(data) + 1), [d.mean() for d in data], marker='D', color='w', edgecolor='k', s=10, zorder=5)
        ax.axhline(0, color='k', lw=0.6, ls='--')
        ax.set_ylabel(f'{STAT_TEX[stat]}\nrelative error (%)', fontsize=7.5)
        ax.grid(True, axis='y', lw=0.3, alpha=0.35)
    axs[-1].set_xticks(range(1, len(REF_IDS) + 1))
    axs[-1].set_xticklabels([lab(s) for s in REF_IDS], rotation=45, ha='right', fontsize=7)
    fig.tight_layout()
    savefig(fig, 'supplementary/FigS3_error_distributions_n10.png')


def figS4():
    """Fig. S4: annual mean flow with Sen trend line and Pettitt change point."""
    fig, axs = grid(len(ORDER), 4, h=1.45)
    for ax, sid in zip(axs, ORDER):
        ys = YS[sid]
        ann = ys.S / ys.C
        ax.plot(ys.years, ann, '-', color=COLORS[sid], lw=0.8)
        b, a = sen_slope(ann)
        ax.plot(ys.years, a + b * np.arange(ys.N), 'k--', lw=0.7)
        r = T_STAT[(T_STAT.GRDC_No == sid) & (T_STAT.Series == 'Annual mean')].iloc[0]
        if r['Pettitt_p'] < 0.05:
            ax.axvline(r['Pettitt_year'], color='gray', ls=':', lw=1)
        ax.set_title(f"{lab(sid)}: MK p={r['MK_p']:.2f}, r1={r['r1']:.2f}", fontsize=7, loc='left', pad=1)
        ax.tick_params(labelsize=7)
    fig.tight_layout(h_pad=0.4, w_pad=0.5)
    savefig(fig, 'supplementary/FigS4_annual_series_trends.png')


def figS5():
    """Fig. S5: RMSE relative to the primary scheme for other block lengths,
    independent years and detrended records."""
    fig, axs = plt.subplots(1, 3, figsize=(6.85, 2.9))
    for ax, stat in zip(axs, ['q_mean', 'ann_max', 'q100']):
        for sid in REF_IDS:
            r0 = RMSE[('main', PRIMARY, 'rel', sid, stat)]
            for sch, ls in [('iid', (0, (1, 1))), ('cbb3', '-'), ('cbb10', '--')]:
                ax.plot(NS, RMSE[('main', sch, 'rel', sid, stat)] / r0, ls=ls, color=COLORS[sid], lw=0.6, alpha=0.6)
            ax.plot(NS, RMSE[('detrended', PRIMARY, 'rel', sid, stat)] / r0, ls='-.', color=COLORS[sid], lw=0.9, alpha=0.8)
        ax.axhline(1, color='k', lw=0.6)
        ax.set_xscale('log'); ax.set_xticks([5, 10, 20, 50]); ax.set_xticklabels(['5', '10', '20', '50'])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_title(STAT_TEX[stat], fontsize=8.5); ax.set_xlabel('n (years)')
        ax.grid(True, lw=0.3, alpha=0.35)
    axs[0].set_ylabel(f'RMSE / RMSE ({SCHEME_LABEL[PRIMARY]})', fontsize=7)
    hand = [Line2D([], [], color='k', ls=(0, (1, 1)), lw=0.8, label='Independent years'),
            Line2D([], [], color='k', ls='-', lw=0.8, label='Blocks L = 3'),
            Line2D([], [], color='k', ls='--', lw=0.8, label='Blocks L = 10'),
            Line2D([], [], color='k', ls='-.', lw=1.0, label=f'Detrended, L = {PRIMARY[3:]}')]
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    fig.legend(handles=hand, loc='lower center', ncol=4, fontsize=7, frameon=False)
    savefig(fig, 'supplementary/FigS5_blocklength_detrending.png')


def figS6():
    """Fig. S6: median and 5-95% range of errors versus record length,
    averaged within each climate group."""
    fig, axs = plt.subplots(2, 3, figsize=(6.85, 4.6), sharex=True)
    axs = axs.ravel()
    d0 = T_SUM[(T_SUM.variant == 'main') & (T_SUM.scheme == PRIMARY)]
    for ax, stat in zip(axs, ['q_mean', 'q10', 'q50', 'q90', 'ann_max', 'q100']):
        meas = 'log' if stat == 'q90' else 'rel'
        for g in GROUPS_PRESENT:
            d = d0[(d0.GRDC_No.isin(members(g))) & (d0.stat == STAT_TXT[stat]) & (d0.measure == meas)]
            a = d.groupby('n')[['P5', 'median', 'P95']].mean()
            ax.fill_between(a.index, a.P5, a.P95, color=GROUP_COL[g], alpha=0.15, lw=0)
            ax.plot(a.index, a['median'], color=GROUP_COL[g], lw=1)
            ax.plot(a.index, a.P5, color=GROUP_COL[g], lw=0.5, ls='--')
            ax.plot(a.index, a.P95, color=GROUP_COL[g], lw=0.5, ls='--')
        ax.axhline(0, color='k', lw=0.5)
        ax.set_xscale('log'); ax.set_xticks([5, 10, 20, 50]); ax.set_xticklabels(['5', '10', '20', '50'])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_title(STAT_TEX[stat] + (' (log-ratio, % equiv.)' if meas == 'log' else ''), fontsize=8)
        ax.grid(True, lw=0.3, alpha=0.35)
    for ax in axs[3:]:
        ax.set_xlabel('n (years)')
    axs[0].set_ylabel('Error (%): median, P5–P95'); axs[3].set_ylabel('Error (%): median, P5–P95')
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    fig.legend(handles=group_handles(marker=None), loc='lower center', ncol=len(GROUPS_PRESENT), fontsize=7, frameon=False)
    savefig(fig, 'supplementary/FigS6_error_percentile_bands.png')


def figS7():
    """Fig. S7: RMSE of mean discharge at n = 10 versus catchment descriptors."""
    fig, axs = plt.subplots(1, 3, figsize=(6.85, 2.7))
    y = np.array([RMSE[('main', PRIMARY, 'rel', s, 'q_mean')][NI[10]] for s in REF_IDS])
    for ax, k in zip(axs, ['CV_annual', 'Area_km2', 'BFI']):
        x = np.asarray(desc[k], float)
        ax.scatter(x, y, c=[COLORS[s] for s in REF_IDS], s=18)
        if k == 'Area_km2':
            ax.set_xscale('log')
        ok = np.isfinite(x) & np.isfinite(y)
        rho, p = stats.spearmanr(x[ok], y[ok])
        ax.text(0.04, 0.92, f'$\\rho_s$ = {rho:.2f} (p = {p:.3f})', transform=ax.transAxes, fontsize=7)
        ax.set_xlabel({'CV_annual': r'$CV$ of annual mean flow', 'Area_km2': 'Catchment area (km²)',
                       'BFI': 'Baseflow index'}[k])
        ax.grid(True, lw=0.3, alpha=0.35)
    axs[0].set_ylabel(r'RMSE of $\bar{Q}$ at n = 10 (%)')
    fig.tight_layout()
    savefig(fig, 'supplementary/FigS7_descriptors.png')


def figS8():
    """Fig. S8: RMSE of mean discharge versus record length for every
    reference catchment and resampling scheme, with analytical curves."""
    fig, axs = grid(len(REF_IDS), 4, h=1.65)
    for ax, sid in zip(axs, REF_IDS):
        for sch in ['sub', 'iid', PRIMARY, 'win']:
            st = SCHEME_STYLE[sch]
            ax.plot(NS, RMSE[('main', sch, 'rel', sid, 'q_mean')], ls=st['ls'], marker=st['marker'],
                    color=st['color'], ms=2, lw=0.8)
        ax.plot(NS, THEORY[sid]['iid'], color='k', lw=0.6, ls='-.')
        ax.plot(NS, THEORY[sid]['sub'], color='k', lw=0.6, ls=':')
        loglog_axes(ax)
        ax.tick_params(labelsize=7)
        ax.set_title(f"{lab(sid)} (N = {NYR[sid]})", fontsize=7, loc='left', pad=1)
    hand = [Line2D([], [], **SCHEME_STYLE[s], ms=3, lw=1, label=SCHEME_LABEL[s]) for s in ['sub', 'iid', PRIMARY, 'win']]
    hand += [Line2D([], [], color='k', ls='-.', lw=0.8, label='Analytical, independent years'),
             Line2D([], [], color='k', ls=':', lw=0.8, label='Analytical, without replacement')]
    fig.tight_layout(rect=(0, 0.05, 1, 1), h_pad=0.4)
    fig.legend(handles=hand, loc='lower center', ncol=3, fontsize=7, frameon=False)
    savefig(fig, 'supplementary/FigS8_schemes_by_catchment.png')


log('\nFigures')
for fn in [fig1, fig2, fig3, fig4, fig5, figS1, figS2, figS3, figS4, figS5, figS6, figS7, figS8]:
    safe(fn)

# ── Saving results ───────────────────────────────────────────────────────────
# All replicate estimates are stored in a compressed archive so that every
# table and figure can be reproduced without repeating the resampling.
np.savez_compressed(f'{OUT}/supplementary/replicate_estimates.npz',
                    **{f'{v}__{s}__{sid}': EST[(v, s, sid)].astype(np.float32) for (v, s, sid) in EST},
                    sample_lengths=np.array(SAMPLE_LENGTHS), stat_keys=np.array(STAT_KEYS))
REFTAB = pd.DataFrame({sid: REF[sid] for sid in ORDER}, index=STAT_KEYS).T
REFTAB.to_csv(f'{OUT}/supplementary/reference_values.csv')
SIG.to_csv(f'{OUT}/supplementary/signatures_all.csv')
pd.DataFrame(DT_INFO).to_csv(f'{OUT}/supplementary/detrending_factors.csv', index=False)
pd.DataFrame([{'scheme': k[1], 'measure': k[2], 'GRDC_No': k[3], 'stat': STAT_TXT[k[4]], 'variant': k[0],
               **{f'n{n}': v for n, v in zip(SAMPLE_LENGTHS, arr)}} for k, arr in RMSE.items()]
             ).to_csv(f'{OUT}/supplementary/RMSE_curves_all.csv', index=False)
pd.DataFrame([{'GRDC_No': s, 'cv0_annual': THEORY[s]['cv0'], 'N': THEORY[s]['N'],
               **{f'iid_n{n}': v for n, v in zip(SAMPLE_LENGTHS, THEORY[s]['iid'])},
               **{f'sub_n{n}': v for n, v in zip(SAMPLE_LENGTHS, THEORY[s]['sub'])}} for s in REF_IDS]
             ).to_csv(f'{OUT}/supplementary/analytical_curves_mean.csv', index=False)
with open(f'{OUT}/run_info.json', 'w') as fh:
    json.dump({'seed': SEED, 'B': B, 'sample_lengths': SAMPLE_LENGTHS, 'complete_frac': COMPLETE_FRAC, 'water_year_start_month': WATER_YEAR_START,
               'block_lengths': BLOCK_LENGTHS, 'primary': PRIMARY, 'min_windows': MIN_WINDOWS,
               'log_delta_frac': LOG_DELTA_FRAC, 'pymannkendall': HAVE_PMK, 'statsmodels': HAVE_SM,
               'numpy': np.__version__, 'pandas': pd.__version__, 'runtime_s': time.time() - T0,
               'stations': {s: {**{k: META[s].get(k) for k in ['river', 'station', 'country', 'lat', 'lon',
                                                                 'area', 'koppen', 'group']}, 'N': NYR[s]} for s in ORDER}},
              fh, indent=1, default=str)

zname = 'streamflow_uncertainty_outputs.zip'
with zipfile.ZipFile(zname, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(OUT):
        for fn in fnames:
            zf.write(os.path.join(root, fn))
log(f'\nTotal runtime: {(time.time() - T0) / 60:.1f} min')
if IN_COLAB:
    colab_files.download(zname)
log(f'Done - {zname} created.')
